### Importing the parsing and data-handling libraries

In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version

print("Python:", sys.version)

packages = ["docling", "pypdf", "python-docx", "python-pptx"]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]
docling: 2.130.0
pypdf: 6.19.0
python-docx: 1.2.0
python-pptx: 1.0.2


In [ ]:
#Docling
import docling
from docling.datamodel.base_models import ConversionStatus, InputFormat
# ConversionStatus: allows us to check if extraction succeeded or failed.
# InputFormat: defines the file types Docling can handle.

from docling.datamodel.pipeline_options import PdfPipelineOptions
# PdfPipelineOptions: configurations for PDF extraction (like preserving tables).

from docling.document_converter import DocumentConverter, PdfFormatOption
# DocumentConverter: the main tool that turns a file into structured data.
# PdfFormatOption: passes the specific PDF settings to the converter.

from docling_core.types.doc import (
    DoclingDocument, TextItem, ListItem, TableItem, PictureItem,
    ListGroup, SectionHeaderItem, TitleItem,
)
# These are the specific building blocks Docling extracts.
# We need them to identify what is a heading (TitleItem/SectionHeaderItem), 
# what is a paragraph (TextItem), a list (ListGroup), or a table (TableItem).
# This is crucial for Steps 5 & 6 (preserving hierarchy and structure).

import docx  # python-docx: Used to create sample DOCX files.
import pptx  # python-pptx: Used to create sample PPTX slides.
import pypdf # pypdf: Used to create sample PDF files.

import os
import sqlite3 # sqlite3: The local database to save parent/child records.
import json    # json: To save Docling's raw output for inspection.
import re      # re (Regex): To clean text and normalize structure.
import uuid    # uuid: To generate stable, unique IDs for chunks.
import hashlib # hashlib: To hash document content 
from pathlib import Path # Path: Modern file path handling.
from datetime import date, datetime, timezone # date/UTC timestamps.

print("Imports loaded successfully.")

Imports loaded successfully.


In [ ]:
import os
from pathlib import Path

DOCS_DIR = Path("./my_dengue_docs")
DOCS_DIR.mkdir(exist_ok=True)
print(f"Documents folder: {DOCS_DIR.resolve()}")

# Fix 5: single authoritative CORPUS_RELEASE; override with env var in CI/staging
CORPUS_RELEASE = os.environ.get("GUARDDENGUE_CORPUS_RELEASE", "v0.1-dev")
print(f"Corpus release  : {CORPUS_RELEASE}")

# ── Fix 1: controlled vocabularies ──────────────────────────────────────────
TOPIC_VOCAB = frozenset({
    "Dengue basics", "Transmission", "Symptoms", "Prevention", "Hydration",
    "Warning signs", "Diagnosis", "Myths", "Children", "Pregnancy",
    "Medication safety", "Outbreak awareness", "Emergency guidance",
})
# GRADE certainty levels only; do NOT mix document-type strings here
EVIDENCE_LEVEL_VOCAB = frozenset({"high", "moderate", "low", "very_low", "ungraded"})

def validate_registry_entry(filename: str, meta: dict) -> None:
    """Raise on missing source, bad/empty topics, bad evidence_level, or invalid ISO date; auto-quarantine missing dates."""
    if not meta.get("source"):
        raise ValueError(f"{filename}: missing required 'source'.")

    topics = meta.get("topic") or []
    if not topics:
        raise ValueError(f"{filename}: 'topic' must contain at least one controlled label.")
    bad_topics = [t for t in topics if t not in TOPIC_VOCAB]
    if bad_topics:
        raise ValueError(
            f"{filename}: unknown topic(s) {bad_topics}.\n"
            f"  Allowed: {sorted(TOPIC_VOCAB)}"
        )

    ev = meta.get("evidence_level", "")
    if ev not in EVIDENCE_LEVEL_VOCAB:
        raise ValueError(
            f"{filename}: evidence_level={ev!r} not in {sorted(EVIDENCE_LEVEL_VOCAB)}.\n"
            "  NOTE: document types (guideline, rct …) are NOT valid evidence_level values."
        )

    # Auto-quarantine missing dates; validate ISO format if present
    pub_date = meta.get("publication_date")
    if not pub_date:
        if meta.get("review_status") not in ("quarantined", "pending_date_resolution"):
            meta["review_status"] = "quarantined"
            print(f"[QUARANTINE] {filename}: publication_date is missing → review_status=quarantined")
    else:
        date.fromisoformat(pub_date)
SOURCE_REGISTRY = {
    "National Guidelines for Clinical Management of Dengue Fever.pdf": {
        "doc_id": "ncvbdc_dengue_guideline_2023",
        "source": "https://ncvbdc.mohfw.gov.in/Doc/National%20Guidelines%20for%20Clinical%20Management%20of%20Dengue%20Fever%202023.pdf",
        "publication_date": None,
        "evidence_level": "ungraded",
        "topic": [
            "Dengue basics", "Transmission", "Symptoms", "Warning signs",
            "Diagnosis", "Hydration", "Children", "Pregnancy",
            "Medication safety", "Emergency guidance",
        ],
        "language": "en",
        "jurisdiction": "India",
        "license": "MoHFW/NCVBDC (verify reuse terms)",
        "review_status": "quarantined",   # blocked until publication_date is confirmed
        "version": "2023 edition",
    },
    "WHO guidelines for clinical management of arboviral diseases.pdf": {
        "doc_id": "who_arboviral_guideline_2025",
        "source": "https://www.who.int/publications/i/item/9789240111110",
        "publication_date": "2025-07-04",
        "evidence_level": "ungraded",
        "topic": [
            "Dengue basics", "Symptoms", "Diagnosis", "Warning signs",
            "Hydration", "Medication safety", "Emergency guidance",
            "Children", "Pregnancy",
        ],
        "language": "en",
        "jurisdiction": "Global",
        "license": "WHO publishing terms (verify license)",
        "review_status": "pending_review",
        "version": "2025 edition",
    },
    "Laboratory testing for dengue virus_interim guidance.pdf": {
        "doc_id": "who_dengue_lab_guidance_2025",
        "source": "https://www.who.int/publications/i/item/B09394",
        "publication_date": "2025-04-28",
        "evidence_level": "ungraded",
        "topic": [
            "Diagnosis", "Dengue basics", "Symptoms", "Warning signs",
            "Children", "Pregnancy", "Transmission", "Outbreak awareness",
        ],
        "language": "en",
        "jurisdiction": "Global",
        "license": "WHO publishing terms (verify license)",
        "review_status": "pending_review",
        "version": "Interim guidance, April 2025",
    },
}

# Validate every entry at definition time (raises immediately on bad data)
for _fname, _meta in SOURCE_REGISTRY.items():
    validate_registry_entry(_fname, _meta)

print(f"Registry validated: {len(SOURCE_REGISTRY)} document(s).")


Documents folder: C:\Users\LENOVO\Desktop\module_2\my_dengue_docs


In [4]:
for file_path in sorted(DOCS_DIR.iterdir()):
    if file_path.is_file():
        print(repr(file_path.name))

'Laboratory testing for dengue virus_interim guidance.pdf'
'National Guidelines for Clinical Management of Dengue Fever.pdf'
'WHO guidelines for clinical management of arboviral diseases.pdf'


In [5]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".pptx", ".md"}

doc_paths = sorted(
    path for path in DOCS_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

if not doc_paths:
    raise FileNotFoundError(f"No supported files found in {DOCS_DIR.resolve()}")

missing_metadata = [
    path.name for path in doc_paths
    if path.name not in SOURCE_REGISTRY
]

if missing_metadata:
    raise KeyError(
        "These filenames do not exactly match SOURCE_REGISTRY keys: "
        f"{missing_metadata}"
    )

registered_documents = [
    (path, SOURCE_REGISTRY[path.name])
    for path in doc_paths
]

print(f"Matched {len(registered_documents)} document(s):")
for path, metadata in registered_documents:
    print(f"- {path.name} -> {metadata['doc_id']}")

Matched 3 document(s):
- Laboratory testing for dengue virus_interim guidance.pdf -> who_dengue_lab_guidance_2025
- National Guidelines for Clinical Management of Dengue Fever.pdf -> ncvbdc_dengue_guideline_2023
- WHO guidelines for clinical management of arboviral diseases.pdf -> who_arboviral_guideline_2025


In [6]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [ ]:
EXTRACTED_DIR = Path("./extracted")
EXTRACTED_DIR.mkdir(exist_ok=True)

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)


In [ ]:
def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
parsed_docs = {}  # {doc_id: {"doc": DoclingDocument, "meta": dict, "out_dir": Path}}

for path, meta in registered_documents:
    doc_id = meta["doc_id"]
    out_dir = EXTRACTED_DIR / doc_id
    out_dir.mkdir(exist_ok=True)
    json_path = out_dir / "document.json"
    doc_hash = file_sha256(path)

    if json_path.exists():
        print(f"[CACHE] Loading {doc_id} from {json_path}")
        doc = DoclingDocument.load_from_json(json_path)
    else:
        print(f"[PARSE] Converting {path.name} ...")
        result = converter.convert(path)
        if result.status != ConversionStatus.SUCCESS:
            raise RuntimeError(f"{doc_id} requires extraction review: {result.status}")
        doc = result.document
        doc.save_as_json(json_path)

        # Save lossless table HTML and original figure crops
        for i, table in enumerate(doc.tables):
            (out_dir / f"table-{i:03d}.html").write_text(
                table.export_to_html(doc=doc), encoding="utf-8"
            )
        for i, picture in enumerate(doc.pictures):
            img = picture.get_image(doc)
            if img is not None:
                img.save(out_dir / f"picture-{i:03d}.png")

    parsed_docs[doc_id] = {
        "doc": doc,
        "meta": {**meta, "content_hash": doc_hash, "filename": path.name},
        "out_dir": out_dir,
    }
    print(f"  -> {doc_id} | pages={len(doc.pages)} | tables={len(doc.tables)} | figures={len(doc.pictures)} | sha256={doc_hash[:12]}...")

In [ ]:
from docling_core.types.doc import ListItem  # Fix 2: import ListItem for list handling

def get_page(item):
    return item.prov[0].page_no if getattr(item, "prov", None) else None

def get_caption(item, doc):
    try:
        return re.sub(r"\s+", " ", item.caption_text(doc) or "").strip()
    except Exception:
        return ""

# Fix 2a: preserve list markers so clinical bullet lists are never merged into blobs
def get_list_text(item, level: int = 0) -> str:
    """Return a ListItem's text with an indented bullet marker retained."""
    marker = "  " * level + "\u2022 "
    text = re.sub(r"\s+", " ", getattr(item, "text", "") or "").strip()
    return f"{marker}{text}" if text else ""

def get_bbox(item):
    if not getattr(item, "prov", None) or getattr(item.prov[0], "bbox", None) is None:
        return None
    b = item.prov[0].bbox
    return b.model_dump() if hasattr(b, "model_dump") else str(b)

def _resolve_text(ann, doc) -> str:
    if isinstance(ann, str):
        return re.sub(r"\s+", " ", ann).strip()
    try:
        resolved = ann.resolve(doc)
        txt = getattr(resolved, "text", "") or ""
        if txt:
            return re.sub(r"\s+", " ", txt).strip()
    except Exception:
        pass
    txt = getattr(ann, "text", "") or ""
    return re.sub(r"\s+", " ", str(txt)).strip() if txt else ""

def get_table_footnotes(item, doc) -> str:
    """
    Return table footnote/annotation texts joined by newlines.
    Deduplicates resolved strings so the same footnote is not repeated when
    both item.annotations and item.footnotes carry the same pointer.
    """
    seen: set[str] = set()
    parts: list[str] = []

    def _add(raw_text: str) -> None:
        t = re.sub(r"\s+", " ", raw_text).strip()
        if t and t not in seen:
            seen.add(t)
            parts.append(f"[footnote] {t}")

    # Primary path: item.annotations (Docling 2.x standard)
    for ann in getattr(item, "annotations", None) or []:
        _add(_resolve_text(ann, doc))

    # Secondary path: item.footnotes (some Docling builds / older API)
    for fn in getattr(item, "footnotes", None) or []:
        _add(_resolve_text(fn, doc))

    return "\n".join(parts)


In [ ]:
for doc_id, bundle in parsed_docs.items():
    doc = bundle["doc"]
    out_dir = bundle["out_dir"]

    counts = {"headings": 0, "text": 0, "lists": 0, "tables": 0, "figures": 0}
    section_stack = []
    headings_log  = []
    tables_log    = []
    figures_log   = []
    t_idx = 0
    f_idx = 0

    for item, level in doc.iterate_items():
        page = get_page(item)

        if isinstance(item, (TitleItem, SectionHeaderItem)):
            txt = re.sub(r"\s+", " ", item.text or "").strip()
            if txt:
                while section_stack and section_stack[-1][0] >= level:
                    section_stack.pop()
                section_stack.append((level, txt))
                counts["headings"] += 1
                headings_log.append((page, level, " > ".join(h for _, h in section_stack)))

        # Fix 5: ListItem BEFORE TextItem — ListItem subclasses TextItem in Docling
        elif isinstance(item, ListItem):
            if get_list_text(item, level).strip():
                counts["lists"] += 1

        elif isinstance(item, TextItem):
            if (item.text or "").strip():
                counts["text"] += 1

        elif isinstance(item, TableItem):
            counts["tables"] += 1
            sec  = " > ".join(h for _, h in section_stack) or "Root"
            rows = getattr(getattr(item, "data", None), "num_rows", "?")
            cols = getattr(getattr(item, "data", None), "num_cols", "?")
            fn   = get_table_footnotes(item, doc).replace("\n", " ")
            cap  = get_caption(item, doc)
            tables_log.append((t_idx, page, f"{rows}x{cols}", sec, cap,
                                fn[:60] if fn else ""))
            t_idx += 1

        elif isinstance(item, PictureItem):
            counts["figures"] += 1
            sec  = " > ".join(h for _, h in section_stack) or "Root"
            img  = item.get_image(doc)
            dims = f"{img.size[0]}x{img.size[1]}" if img else "missing"
            figures_log.append((f_idx, page, dims, sec, get_caption(item, doc)))
            f_idx += 1

    print("=" * 80)
    print(f"DOC: {doc_id} ({bundle['meta']['filename']})")
    print(f"Pages: {len(doc.pages)} | Counts: {counts}")

    print("\n-- Headings (first 15) --")
    for pg, lvl, sp in headings_log[:15]:
        print(f"  p.{str(pg):>3} | L{lvl} | {sp}")

    print(f"\n-- Tables ({len(tables_log)} total, showing first 10) --")
    for idx, pg, shape, sec, cap, fn in tables_log[:10]:
        fn_tag = f" | fn: {fn}" if fn else ""
        print(f"  table-{idx:03d} | p.{str(pg):>3} | {shape:<7} | sec: {sec[:40]} | cap: {cap[:40]}{fn_tag}")

    print(f"\n-- Figures ({len(figures_log)} total, showing first 15) --")
    for idx, pg, dims, sec, cap in figures_log[:15]:
        print(f"  picture-{idx:03d} | p.{str(pg):>3} | {dims:<9} | sec: {sec[:40]} | cap: {cap[:40]}")


In [ ]:
def inspect_page(doc_id: str, target_page: int):
    doc = parsed_docs[doc_id]["doc"]
    print(f"=== {doc_id} | Page {target_page} ===")
    found = 0
    for item, level in doc.iterate_items():
        if get_page(item) != target_page:
            continue
        found += 1
        kind = type(item).__name__
        if isinstance(item, TableItem):
            fn = get_table_footnotes(item, doc).replace("\n", " ")
            preview = item.export_to_markdown(doc=doc)[:200].replace("\n", " | ")
            if fn:
                preview += f" || {fn[:100]}"
        elif isinstance(item, PictureItem):
            img = item.get_image(doc)
            dims = f"{img.size[0]}x{img.size[1]}" if img else "missing"
            preview = f"[IMAGE {dims}] caption={get_caption(item, doc)!r}"
        elif isinstance(item, ListItem):
            preview = get_list_text(item, level)[:160]
        else:
            preview = re.sub(r"\s+", " ", getattr(item, "text", "") or "").strip()[:160]
        print(f"[{kind:<17} L{level}] {preview}")
    if not found:
        print("No extracted items on this page.")


inspect_page("who_arboviral_guideline_2025", 10)

In [ ]:
from PIL import Image

MIN_FIGURE_PX = 120  # Skip small icons, logos, or decorative lines

figure_inventory = []

for doc_id, bundle in parsed_docs.items():
    doc = bundle["doc"]
    out_dir = bundle["out_dir"]
    section_stack = []
    fallback_idx = 0

    for item, level in doc.iterate_items():
        if isinstance(item, (TitleItem, SectionHeaderItem)):
            txt = re.sub(r"\s+", " ", item.text or "").strip()
            if txt:
                while section_stack and section_stack[-1][0] >= level:
                    section_stack.pop()
                section_stack.append((level, txt))

        elif isinstance(item, PictureItem):
            ref = getattr(item, "self_ref", "")
            if ref.startswith("#/pictures/"):
                pic_idx = int(ref.split("/")[-1])
            else:
                pic_idx = fallback_idx
            fallback_idx += 1

            img_path = out_dir / f"picture-{pic_idx:03d}.png"
            w, h = (0, 0)
            if not img_path.exists():
                status = "missing_image"
                is_relevant = False
            else:
                with Image.open(img_path) as im:
                    w, h = im.size
                if w < MIN_FIGURE_PX or h < MIN_FIGURE_PX:
                    status = "skipped_small"
                    is_relevant = False
                else:
                    status = "pending_vlm"
                    is_relevant = True

            figure_inventory.append({
                "figure_id": f"{doc_id}__fig_{pic_idx:03d}",
                "doc_id": doc_id,
                "source_pdf": bundle["meta"]["filename"],
                "source": bundle["meta"]["source"],
                "page": get_page(item),
                "bbox": get_bbox(item),
                "section_path": " > ".join(h for _, h in section_stack) or "Document Root",
                "caption": get_caption(item, doc),
                "image_path": str(img_path),
                "width": w,
                "height": h,
                "is_relevant": is_relevant,
                "review_status": status,
            })

candidates = [f for f in figure_inventory if f["review_status"] == "pending_vlm"]
skipped_small = sum(1 for f in figure_inventory if f["review_status"] == "skipped_small")
missing_imgs = sum(1 for f in figure_inventory if f["review_status"] == "missing_image")

print(
    f"Total figures: {len(figure_inventory)} | "
    f"Missing: {missing_imgs} | "
    f"Skipped (<{MIN_FIGURE_PX}px): {skipped_small} | "
    f"Ready for Gemini: {len(candidates)}"
)

In [ ]:
from google import genai
from google.genai import types

# Pin an explicit Gemini model version (avoid unversioned aliases)
VLM_MODEL = "gemini-2.5-flash"
TEST_LIMIT = 3  # Start with 3 figures to verify output quality; set to None to process all

FIGURE_PROMPT = """
You are digitizing a figure from a dengue clinical guideline for offline review.
Do NOT invent clinical advice or extrapolate beyond what is literally visible.
Set "is_relevant" to false for logos, cover art, or purely decorative graphics.
Return STRICT JSON:
{
  "is_relevant": true,
  "figure_type": "flowchart | clinical_chart | diagram | table_image | map | decorative",
  "description": "Faithful text description preserving exact thresholds, units, populations, and qualifiers.",
  "structured_logic": {
    "nodes": [{"id": "n1", "label": "exact node text"}],
    "edges": [{"from": "n1", "to": "n2", "condition": "branch label or condition"}]
  },
  "ambiguous_elements": ["List any blurry, cut-off, or unreadable labels/arrows; empty if none"]
}
"""

FIGURES_MANIFEST_PATH = EXTRACTED_DIR / "figures_manifest.json"

# Load cached results if this cell was run earlier
if FIGURES_MANIFEST_PATH.exists():
    saved_records = {
        r["figure_id"]: r
        for r in json.loads(FIGURES_MANIFEST_PATH.read_text(encoding="utf-8"))
    }
else:
    saved_records = {}

client_genai = None
processed_now = 0

for fig in figure_inventory:
    fid = fig["figure_id"]

    # Skip if already processed (still retries pending_vlm and vlm_error)
    if fid in saved_records and saved_records[fid].get("review_status") not in ("pending_vlm", "vlm_error"):
        continue

    # Save small or missing items directly without calling Gemini
    if fig["review_status"] in ("skipped_small", "missing_image"):
        saved_records[fid] = {
            **fig,
            "vlm_model": None,
            "figure_type": None,
            "description": None,
            "structured_logic": {"nodes": [], "edges": []},
            "ambiguous_elements": [],
        }
        continue

    if TEST_LIMIT is not None and processed_now >= TEST_LIMIT:
        break

    # Lazy-init client so fully cached runs don't fail if GEMINI_API_KEY isn't set
    if client_genai is None:
        api_key = os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise EnvironmentError("GEMINI_API_KEY is not set in environment variables.")
        client_genai = genai.Client(api_key=api_key)

    context_hint = (
        f"Document: {fig['doc_id']} | Page: {fig['page']} | "
        f"Section: {fig['section_path']} | Caption: {fig['caption'] or 'None'}"
    )

    try:
        with Image.open(fig["image_path"]) as pil_img:
            response = client_genai.models.generate_content(
                model=VLM_MODEL,
                contents=[FIGURE_PROMPT, context_hint, pil_img],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0,
                ),
            )
        if not response.text:
            raise ValueError("Empty response.text from Gemini (possible safety filter).")
        vlm_out = json.loads(response.text)
        if not isinstance(vlm_out, dict):
            raise ValueError(f"Expected JSON object, got {type(vlm_out).__name__}")
    except Exception as e:
        saved_records[fid] = {
            **fig,
            "vlm_model": VLM_MODEL,
            "review_status": "vlm_error",
            "error": str(e),
            "extracted_at": datetime.now(timezone.utc).isoformat(),
        }
        FIGURES_MANIFEST_PATH.write_text(
            json.dumps(list(saved_records.values()), indent=2), encoding="utf-8"
        )
        print(f"[ERROR] {fid} (p.{fig['page']}): {e}")
        continue

    is_rel = bool(vlm_out.get("is_relevant", True))

    saved_records[fid] = {
        **fig,
        "is_relevant": is_rel,
        "figure_type": vlm_out.get("figure_type", "unknown"),
        "description": vlm_out.get("description", ""),
        "structured_logic": vlm_out.get("structured_logic", {"nodes": [], "edges": []}),
        "ambiguous_elements": vlm_out.get("ambiguous_elements", []),
        "vlm_model": VLM_MODEL,
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "review_status": "pending_review" if is_rel else "skipped_decorative",
    }

    # Checkpoint after every figure so progress isn't lost on interruption
    FIGURES_MANIFEST_PATH.write_text(
        json.dumps(list(saved_records.values()), indent=2), encoding="utf-8"
    )
    processed_now += 1
    print(f"[{processed_now}] {fid} (p.{fig['page']}) -> relevant={is_rel} | type={saved_records[fid]['figure_type']}")

# Final flush so skipped_small and missing_image records are always saved
FIGURES_MANIFEST_PATH.write_text(
    json.dumps(list(saved_records.values()), indent=2), encoding="utf-8"
)
print(f"\nSaved {len(saved_records)} figure records to {FIGURES_MANIFEST_PATH}")

In [ ]:
# Set True ONCE after switching to uuid5 chunk_ids to wipe old positional IDs.
# After the first clean run flip back to False — cached Docling artifacts are NEVER deleted.
RESET_STALE_STORE = True

import shutil

_db   = EXTRACTED_DIR / "guarddengue_chunks.db"
_qdir = EXTRACTED_DIR / "qdrant_local"

if RESET_STALE_STORE:
    deleted = []
    try:
        for db_file in (
            _db,
            Path(f"{_db}-wal"),
            Path(f"{_db}-shm"),
            Path(f"{_db}-journal"),
        ):
            if db_file.exists():
                db_file.unlink()
                deleted.append(str(db_file))

        if _qdir.exists():
            shutil.rmtree(_qdir)
            deleted.append(str(_qdir))
    except PermissionError as e:
        raise RuntimeError(
            f"Store file is still locked ({e}). Close open SQLite/Qdrant connections or restart the kernel, then re-run."
        ) from e

    if deleted:
        print("[RESET] Deleted stale stores:")
        for d in deleted:
            print(f"  {d}")
        print("  (cached document.json / table-*.html / picture-*.png / figures_manifest.json preserved)")
    else:
        print("[RESET] Nothing to delete — stores were already absent.")
else:
    print("[RESET] Skipped (RESET_STALE_STORE=False). Flip to True to wipe stale chunk_ids.")

In [ ]:
import html
from llama_index.core.schema import NodeRelationship, RelatedNodeInfo, TextNode

MAX_PROSE_CHARS = 1200   # Split long prose; never split tables/figures
DB_PATH = EXTRACTED_DIR / "guarddengue_chunks.db"

# Load figure descriptions from Step 2
figures_by_id = {}
if FIGURES_MANIFEST_PATH.exists():
    for rec in json.loads(FIGURES_MANIFEST_PATH.read_text(encoding="utf-8")):
        figures_by_id[rec["figure_id"]] = rec

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript("""
CREATE TABLE IF NOT EXISTS parent_chunks (
    parent_id      TEXT PRIMARY KEY,
    doc_id         TEXT NOT NULL,
    section_path   TEXT NOT NULL,
    page_start     INTEGER,
    page_end       INTEGER,
    text           TEXT NOT NULL,
    metadata_json  TEXT NOT NULL,
    review_status  TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS child_chunks (
    chunk_id           TEXT PRIMARY KEY,
    parent_id          TEXT NOT NULL REFERENCES parent_chunks(parent_id),
    doc_id             TEXT NOT NULL,
    element_type       TEXT NOT NULL,
    section_path       TEXT NOT NULL,
    page_start         INTEGER,
    page_end           INTEGER,
    bbox_json          TEXT,
    artifact_ref       TEXT,
    artifact_path      TEXT,
    text               TEXT NOT NULL,
    metadata_json      TEXT NOT NULL,
    review_status      TEXT NOT NULL,
    chunk_content_hash TEXT,          -- Fix 3: SHA-256 of normalised chunk text
    reviewed_at        TEXT,          -- Fix 3: UTC ISO timestamp set on approval; NULL until then
    superseded_by      TEXT           -- Fix 3: chunk_id of replacement; NULL = current
);
""")
conn.commit()

# Fix 3: safe column-add for existing DBs (idempotent ALTER TABLE)
for col, ddl in [
    ("chunk_content_hash", "TEXT"),
    ("reviewed_at",        "TEXT"),
    ("superseded_by",      "TEXT"),
]:
    try:
        conn.execute(f"ALTER TABLE child_chunks ADD COLUMN {col} {ddl}")
        conn.commit()
        print(f"[SCHEMA] Added column child_chunks.{col}")
    except Exception:
        pass   # column already exists


def chunk_content_sha256(text: str) -> str:
    """SHA-256 of normalised (collapsed-whitespace) chunk text."""
    normalised = re.sub(r"\s+", " ", text).strip()
    return hashlib.sha256(normalised.encode("utf-8")).hexdigest()


def make_chunk_id(doc_id: str, section_path: str, element_type: str,
                  content_hash: str, occurrence_idx: int) -> str:
    """Fix 3: deterministic uuid5 anchored to content; collision-safe on repeated text."""
    seed = f"{doc_id}|{section_path}|{element_type}|{content_hash}|{occurrence_idx}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, seed))


def extract_ref_index(item, prefix: str, fallback: int) -> int:
    ref = getattr(item, "self_ref", "")
    if ref.startswith(prefix):
        return int(ref.split("/")[-1])
    return fallback


def build_parent_child_chunks(doc_id: str, bundle: dict) -> list:
    doc     = bundle["doc"]
    meta    = bundle["meta"]
    out_dir = bundle["out_dir"]

    base_meta = {
        "doc_id":           doc_id,
        "filename":         meta["filename"],
        "source":           meta["source"],
        "publication_date": meta["publication_date"],
        "evidence_level":   meta["evidence_level"],
        "topic":            meta["topic"],
        "language":         meta["language"],
        "jurisdiction":     meta["jurisdiction"],
        "license":          meta["license"],
        "document_version": meta["version"],
        "content_hash":     meta["content_hash"],
        "corpus_release":   CORPUS_RELEASE,
        "reviewed_at":      None,       # Fix 3: set to UTC timestamp on approval
        "superseded_by":    None,       # Fix 3: null = current/active chunk
    }
    doc_status = meta["review_status"]

    # 1. Group items in reading order under their section heading
    sections = []
    current  = {"section_path": "Document Root", "items": [], "pages": []}
    section_stack = []
    t_fallback = f_fallback = 0
    _occurrence_counters: dict = {}   # (section_path, element_type, content_hash) -> count

    for item, level in doc.iterate_items():
        page = get_page(item)
        bbox = get_bbox(item)

        if isinstance(item, (TitleItem, SectionHeaderItem)):
            heading = re.sub(r"\s+", " ", item.text or "").strip()
            if heading:
                if current["items"]:
                    sections.append(current)
                while section_stack and section_stack[-1][0] >= level:
                    section_stack.pop()
                section_stack.append((level, heading))
                current = {
                    "section_path": " > ".join(h for _, h in section_stack),
                    "items": [],
                    "pages": [page] if page else [],
                }

        # Fix 5: ListItem BEFORE TextItem — ListItem subclasses TextItem in Docling
        elif isinstance(item, ListItem):
            txt = get_list_text(item, level)
            if txt.strip():
                current["items"].append({
                    "type": "prose", "text": txt, "page": page,
                    "bbox": bbox, "artifact_ref": None, "artifact_path": None,
                    "extra_meta": {}, "item_status": doc_status,
                })
                if page:
                    current["pages"].append(page)

        elif isinstance(item, TextItem):
            txt = re.sub(r"\s+", " ", item.text or "").strip()
            if txt:
                current["items"].append({
                    "type": "prose", "text": txt, "page": page,
                    "bbox": bbox, "artifact_ref": None, "artifact_path": None,
                    "extra_meta": {}, "item_status": doc_status,
                })
                if page:
                    current["pages"].append(page)

        elif isinstance(item, TableItem):
            t_idx = extract_ref_index(item, "#/tables/", t_fallback)
            t_fallback += 1
            table_id  = f"{doc_id}__table_{t_idx:03d}"
            html_path = str(out_dir / f"table-{t_idx:03d}.html")
            caption   = get_caption(item, doc)
            fn_text   = get_table_footnotes(item, doc)   # Fix 2b: capture footnotes
            md_text   = item.export_to_markdown(doc=doc).strip()

            if md_text:
                # Append footnotes to both the index text and the saved HTML
                fn_block = f"\n{fn_text}" if fn_text else ""
                block = f"[TABLE {table_id}] {caption}\n{md_text}{fn_block}".strip()

                # Fix 2b: also write escaped footnotes into the saved HTML artifact
                html_full_path = Path(html_path)
                if html_full_path.exists():
                    existing_html = html_full_path.read_text(encoding="utf-8")
                    if fn_text and "[footnote]" not in existing_html:
                        fn_html = "\n".join(
                            f"<p class='table-footnote'>{html.escape(fn)}</p>"
                            for fn in fn_text.split("\n") if fn
                        )
                        html_full_path.write_text(
                            existing_html + f"\n<div class='footnotes'>{fn_html}</div>",
                            encoding="utf-8",
                        )

                current["items"].append({
                    "type": "table", "text": block, "page": page, "bbox": bbox,
                    "artifact_ref": table_id, "artifact_path": html_path,
                    "extra_meta": {"caption": caption, "table_footnotes": fn_text},
                    "item_status": doc_status,
                })
                if page:
                    current["pages"].append(page)

        elif isinstance(item, PictureItem):
            f_idx  = extract_ref_index(item, "#/pictures/", f_fallback)
            f_fallback += 1
            fig_id = f"{doc_id}__fig_{f_idx:03d}"
            fig    = figures_by_id.get(fig_id)
            if fig and fig.get("is_relevant") and fig.get("description"):
                logic   = fig.get("structured_logic") or {}
                nodes_s = "; ".join(
                    n.get("label", "") for n in (logic.get("nodes") or [])
                    if isinstance(n, dict) and n.get("label")
                )
                edges_s = "; ".join(
                    f"{e.get('from', '?')}→{e.get('to', '?')} ({e.get('condition', '')})"
                    for e in (logic.get("edges") or [])
                    if isinstance(e, dict)
                )
                ambi    = "; ".join(str(a) for a in (fig.get("ambiguous_elements") or []))
                desc    = fig["description"]
                fig_text = (
                    f"[FIGURE {fig_id}] {fig.get('caption', '')}\n"
                    f"Type: {fig.get('figure_type', 'unknown')}\n"
                    f"Description: {desc}\n"
                    + (f"Nodes: {nodes_s}\n" if nodes_s else "")
                    + (f"Edges: {edges_s}\n" if edges_s else "")
                    + (f"Ambiguous: {ambi}" if ambi else "")
                ).strip()
                current["items"].append({
                    "type": "figure", "text": fig_text, "page": page, "bbox": bbox,
                    "artifact_ref": fig_id,
                    "artifact_path": fig.get("image_path"),
                    "extra_meta": {
                        "caption":            fig.get("caption", ""),
                        "figure_type":        fig.get("figure_type", "unknown"),
                        "ambiguous_elements": fig.get("ambiguous_elements") or [],
                    },
                    # Keep quarantined docs quarantined; otherwise hold figures until approve_figure()
                    "item_status": "quarantined" if doc_status == "quarantined" else "pending_review",
                })
                if page:
                    current["pages"].append(page)

    if current["items"]:
        sections.append(current)

    llama_nodes = []

    for sec in sections:
        if not sec["items"]:
            continue
        pages   = [p for p in sec["pages"] if p is not None]
        p_start = min(pages) if pages else None
        p_end   = max(pages) if pages else None

        parent_text = (
            f"Section: {sec['section_path']}\n\n"
            + "\n\n".join(x["text"] for x in sec["items"])
        )
        parent_hash = chunk_content_sha256(parent_text)
        p_occ_key   = (sec["section_path"], "parent", parent_hash)
        _occurrence_counters[p_occ_key] = _occurrence_counters.get(p_occ_key, -1) + 1
        parent_id   = make_chunk_id(doc_id, sec["section_path"], "parent",
                                    parent_hash, _occurrence_counters[p_occ_key])

        parent_meta = {
            **base_meta,
            "parent_id":     parent_id,
            "section_path":  sec["section_path"],
            "page_start":    p_start,
            "page_end":      p_end,
            "review_status": doc_status,
        }
        conn.execute("""
            INSERT INTO parent_chunks
            (parent_id, doc_id, section_path, page_start, page_end, text, metadata_json, review_status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(parent_id) DO UPDATE SET
                doc_id        = excluded.doc_id,
                section_path  = excluded.section_path,
                page_start    = excluded.page_start,
                page_end      = excluded.page_end,
                text          = excluded.text,
                metadata_json = excluded.metadata_json,
                review_status = excluded.review_status
        """, (
            parent_id, doc_id, sec["section_path"], p_start, p_end,
            parent_text, json.dumps(parent_meta), doc_status,
        ))

        # Group adjacent prose <= MAX_PROSE_CHARS; tables/figures are always atomic
        children    = []
        prose_buf   = []
        prose_pages = []

        def flush_prose():
            if not prose_buf:
                return
            children.append({
                "type": "prose",
                "text": f"Section: {sec['section_path']}\n" + "\n".join(prose_buf),
                "page_start": min(prose_pages) if prose_pages else p_start,
                "page_end":   max(prose_pages) if prose_pages else p_end,
                "bbox": None, "artifact_ref": None, "artifact_path": None,
                "extra_meta": {}, "review_status": doc_status,
            })
            prose_buf.clear()
            prose_pages.clear()

        for entry in sec["items"]:
            if entry["type"] == "prose":
                if prose_buf and (sum(len(t) for t in prose_buf) + len(entry["text"]) > MAX_PROSE_CHARS):
                    flush_prose()
                prose_buf.append(entry["text"])
                if entry["page"] is not None:
                    prose_pages.append(entry["page"])
            else:
                flush_prose()
                children.append({
                    "type":          entry["type"],
                    "text":          f"Section: {sec['section_path']}\n{entry['text']}",
                    "page_start":    entry["page"],
                    "page_end":      entry["page"],
                    "bbox":          entry["bbox"],
                    "artifact_ref":  entry["artifact_ref"],
                    "artifact_path": entry["artifact_path"],
                    "extra_meta":    entry["extra_meta"],
                    "review_status": entry["item_status"],
                })
        flush_prose()

        prev_chunk_id = None
        for ch in children:
            # Fix 3: collision-safe, content-anchored chunk_id
            cch = chunk_content_sha256(ch["text"])
            occ_key = (sec["section_path"], ch["type"], cch)
            _occurrence_counters[occ_key] = _occurrence_counters.get(occ_key, -1) + 1
            chunk_id = make_chunk_id(doc_id, sec["section_path"], ch["type"],
                                     cch, _occurrence_counters[occ_key])

            child_meta = {
                **base_meta,
                "chunk_id":           chunk_id,
                "parent_id":          parent_id,
                "element_type":       ch["type"],
                "section_path":       sec["section_path"],
                "page_start":         ch["page_start"],
                "page_end":           ch["page_end"],
                "artifact_ref":       ch["artifact_ref"],
                "artifact_path":      ch["artifact_path"],
                "review_status":      ch["review_status"],
                "chunk_content_hash": cch,     # Fix 3
                "reviewed_at":        None,    # Fix 3: set at approval time
                "superseded_by":      None,    # Fix 3: null = active
                **ch["extra_meta"],
            }

            conn.execute("""
                INSERT INTO child_chunks
                (chunk_id, parent_id, doc_id, element_type, section_path, page_start,
                 page_end, bbox_json, artifact_ref, artifact_path, text, metadata_json,
                 review_status, chunk_content_hash, reviewed_at, superseded_by)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                ON CONFLICT(chunk_id) DO UPDATE SET
                    parent_id          = excluded.parent_id,
                    doc_id             = excluded.doc_id,
                    element_type       = excluded.element_type,
                    section_path       = excluded.section_path,
                    page_start         = excluded.page_start,
                    page_end           = excluded.page_end,
                    bbox_json          = excluded.bbox_json,
                    artifact_ref       = excluded.artifact_ref,
                    artifact_path      = excluded.artifact_path,
                    text               = excluded.text,
                    metadata_json      = excluded.metadata_json,
                    review_status      = excluded.review_status,
                    chunk_content_hash = excluded.chunk_content_hash,
                    reviewed_at        = COALESCE(child_chunks.reviewed_at, excluded.reviewed_at),
                    superseded_by      = COALESCE(child_chunks.superseded_by, excluded.superseded_by)
            """, (
                chunk_id, parent_id, doc_id, ch["type"], sec["section_path"],
                ch["page_start"], ch["page_end"], json.dumps(ch["bbox"]),
                ch["artifact_ref"], ch["artifact_path"], ch["text"],
                json.dumps(child_meta), ch["review_status"],
                cch, None, None,
            ))

            node = TextNode(id_=chunk_id, text=ch["text"], metadata=child_meta)
            node.relationships[NodeRelationship.PARENT] = RelatedNodeInfo(node_id=parent_id)
            if prev_chunk_id:
                node.relationships[NodeRelationship.PREVIOUS] = RelatedNodeInfo(node_id=prev_chunk_id)
                llama_nodes[-1].relationships[NodeRelationship.NEXT] = RelatedNodeInfo(node_id=chunk_id)

            llama_nodes.append(node)
            prev_chunk_id = chunk_id

    conn.commit()
    return llama_nodes


all_child_nodes = []
for doc_id, bundle in parsed_docs.items():
    nodes = build_parent_child_chunks(doc_id, bundle)
    all_child_nodes.extend(nodes)
    print(f"[{doc_id}] built {len(nodes)} child chunks.")

print(f"\nTotal child chunks saved to {DB_PATH}: {len(all_child_nodes)}")

In [ ]:
# Fix 5: check_release_sync() – assert all stores share the same CORPUS_RELEASE

def check_release_sync(verbose: bool = True) -> bool:
    """
    Verify SQLite, Qdrant payloads, the local policy artifact, and (if online) Neo4j
    all carry the same CORPUS_RELEASE tag.  Returns True on full agreement.
    """
    ok = True
    tag = CORPUS_RELEASE

    # 1. SQLite: every active approved chunk must carry the current release (NULL-safe)
    total_approved, bad_sqlite = conn.execute("""
        SELECT
            COUNT(*),
            COALESCE(SUM(
                CASE WHEN json_extract(metadata_json, '$.corpus_release') IS NOT ? THEN 1 ELSE 0 END
            ), 0)
        FROM child_chunks
        WHERE review_status = 'approved'
          AND superseded_by IS NULL
    """, (tag,)).fetchone()
    if bad_sqlite:
        print(f"[FAIL] SQLite: {bad_sqlite}/{total_approved} active approved chunk(s) missing or mismatched corpus_release.")
        ok = False
    elif total_approved == 0:
        print(f"[WARN] SQLite: 0 active approved chunks yet (nothing to verify for {tag!r}).")
    elif verbose:
        print(f"[OK]   SQLite: all {total_approved} active approved chunk(s) tagged {tag!r}.")

    # 2. Qdrant: full paginated scan for missing or mismatched release tags
    try:
        total_q, bad_q = 0, 0
        offset = None
        while True:
            points, offset = qdrant.scroll(
                collection_name=COLLECTION_NAME,
                limit=256,
                offset=offset,
                with_payload=True,
                with_vectors=False,
            )
            total_q += len(points)
            bad_q += sum(1 for p in points if (p.payload or {}).get("corpus_release") != tag)
            if not points or offset is None:
                break
        if bad_q:
            print(f"[FAIL] Qdrant: {bad_q}/{total_q} point(s) missing or mismatched corpus_release.")
            ok = False
        elif verbose:
            print(f"[OK]   Qdrant: checked all {total_q} point(s), all tagged {tag!r}.")
    except Exception as exc:
        print(f"[SKIP] Qdrant check skipped ({exc}).")

    # 3. Local policy artifact (supports either a top-level list or {"rules": [...]})
    policy_path = EXTRACTED_DIR / f"safety_policy_{tag}.json"
    if not policy_path.exists():
        # Tolerate absence during early prototyping; warn only
        print(f"[WARN] Policy artifact not yet written: {policy_path.name}")
    else:
        policy_data = json.loads(policy_path.read_text(encoding="utf-8"))
        rules = policy_data.get("rules", [policy_data]) if isinstance(policy_data, dict) else policy_data
        bad_p = [r for r in rules if not isinstance(r, dict) or r.get("corpus_release") != tag]
        if bad_p:
            print(f"[FAIL] Policy artifact: {len(bad_p)} rule(s) missing or mismatched corpus_release.")
            ok = False
        elif verbose:
            print(f"[OK]   Policy artifact: {len(rules)} rule(s) tagged {tag!r}.")

    # 4. Neo4j (optional; skip gracefully if offline)
    try:
        neo4j_pwd = os.environ.get("NEO4J_PASSWORD", "")
        if neo4j_pwd:
            from neo4j import GraphDatabase as _GD
            with _GD.driver(
                os.environ.get("NEO4J_URI", "bolt://localhost:7687"),
                auth=(os.environ.get("NEO4J_USER", "neo4j"), neo4j_pwd),
            ) as _drv:
                recs, _, _ = _drv.execute_query(
                    "MATCH ()-[r:AVOID_IN]->() "
                    "WHERE r.corpus_release IS NULL OR r.corpus_release <> $tag "
                    "RETURN count(r) AS n",
                    tag=tag,
                )
                bad_n = recs[0]["n"] if recs else 0
                if bad_n:
                    print(f"[FAIL] Neo4j: {bad_n} AVOID_IN edge(s) missing or mismatched corpus_release.")
                    ok = False
                elif verbose:
                    print(f"[OK]   Neo4j: all AVOID_IN edges tagged {tag!r}.")
        else:
            if verbose:
                print("[SKIP] Neo4j check skipped (NEO4J_PASSWORD not set).")
    except Exception as exc:
        if verbose:
            print(f"[SKIP] Neo4j check skipped ({exc}).")

    print("\nRelease sync:", "PASS ✓" if ok else "FAIL ✗  — investigate before indexing.")
    return ok


check_release_sync()

In [ ]:
print("=== Active Chunk Counts by Document, Type & Status ===")
rows = conn.execute("""
    SELECT doc_id, element_type, review_status, COUNT(*)
    FROM child_chunks
    WHERE superseded_by IS NULL
    GROUP BY doc_id, element_type, review_status
    ORDER BY doc_id, element_type, review_status
""").fetchall()
for doc_id, el_type, status, cnt in rows:
    print(f"  {doc_id:<32} | {el_type:<7} | {status:<15} | {cnt:>4}")

# Inspect a parent section that contains an active table or figure (if available)
sample_parent = conn.execute("""
    SELECT DISTINCT p.parent_id, p.doc_id, p.section_path, p.page_start, p.page_end
    FROM parent_chunks p
    JOIN child_chunks c ON p.parent_id = c.parent_id
    WHERE c.element_type IN ('figure', 'table')
      AND c.superseded_by IS NULL
    LIMIT 1
""").fetchone()

if sample_parent:
    pid, did, spath, p1, p2 = sample_parent
    print(f"\n=== Sample Parent: {pid} ===")
    print(f"Doc: {did} | Pages: {p1}-{p2} | Section: {spath}")

    children = conn.execute("""
        SELECT chunk_id, element_type, page_start, artifact_path, review_status, substr(text, 1, 180)
        FROM child_chunks
        WHERE parent_id = ?
          AND superseded_by IS NULL
        ORDER BY rowid
    """, (pid,)).fetchall()

    for cid, el_type, pg, art, status, preview in children:
        print(f"\n  -> {cid} | type={el_type} | p.{pg} | status={status}")
        if art:
            print(f"     artifact: {art}")
        print(f"     preview : {preview.replace(chr(10), ' ')}...")

In [ ]:
%pip install qdrant-client fastembed

In [ ]:
import hashlib
from datetime import datetime, timezone

# 1. Provision persistent HITL governance tables in SQLite
conn.executescript("""
CREATE TABLE IF NOT EXISTS review_tasks (
    task_id        TEXT PRIMARY KEY,
    object_id      TEXT NOT NULL,
    object_type    TEXT NOT NULL,          -- 'document_text_tables' | 'figure' | 'table'
    object_hash    TEXT NOT NULL,
    corpus_release TEXT NOT NULL,
    assigned_role  TEXT NOT NULL DEFAULT 'clinical_reviewer',
    status         TEXT NOT NULL,          -- 'quarantined' | 'pending_review' | 'corrected_pending_review' | 'approved' | 'rejected'
    created_at     TEXT NOT NULL,
    updated_at     TEXT NOT NULL,
    UNIQUE(object_id, object_hash, corpus_release)
);

CREATE TABLE IF NOT EXISTS derived_artifacts (
    artifact_id     TEXT PRIMARY KEY,
    source_chunk_id TEXT NOT NULL,
    prior_hash      TEXT NOT NULL,
    corrected_text  TEXT NOT NULL,
    resulting_hash  TEXT NOT NULL,
    created_by      TEXT NOT NULL,
    created_at      TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS review_events (
    event_id              TEXT PRIMARY KEY,
    task_id               TEXT NOT NULL,
    object_id             TEXT NOT NULL,
    reviewer_id           TEXT NOT NULL,
    decision              TEXT NOT NULL,   -- 'release_from_quarantine' | 'approve' | 'reject' | 'correct'
    rationale             TEXT NOT NULL,
    rejection_reason      TEXT,
    prior_hash            TEXT NOT NULL,
    resulting_hash        TEXT NOT NULL,
    corrected_artifact_id TEXT,
    created_at            TEXT NOT NULL
);
""")
conn.commit()


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def _sha256_text(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()


def _ensure_review_task(object_id: str, object_type: str, object_hash: str, status: str) -> str:
    """Idempotently create or fetch a review task bound to an exact object_hash."""
    row = conn.execute("""
        SELECT task_id FROM review_tasks
        WHERE object_id = ? AND object_hash = ? AND corpus_release = ?
    """, (object_id, object_hash, CORPUS_RELEASE)).fetchone()
    if row:
        return row[0]

    now = _utc_now()
    task_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f"task:{CORPUS_RELEASE}:{object_id}:{object_hash}"))
    conn.execute("""
        INSERT INTO review_tasks
            (task_id, object_id, object_type, object_hash, corpus_release, status, created_at, updated_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (task_id, object_id, object_type, object_hash, CORPUS_RELEASE, status, now, now))
    return task_id


def _append_review_event(
    task_id: str,
    object_id: str,
    reviewer_id: str,
    decision: str,
    rationale: str,
    prior_hash: str,
    resulting_hash: str,
    rejection_reason: str | None = None,
    corrected_artifact_id: str | None = None,
) -> str:
    """Append an immutable audit record to review_events."""
    if not reviewer_id or reviewer_id in ("pending_clinical_signoff", "TBD", "prototype"):
        raise ValueError(f"Valid named reviewer_id is required (got {reviewer_id!r}).")
    if not rationale or not rationale.strip():
        raise ValueError("A non-empty rationale is required for every review decision.")

    event_id = str(uuid.uuid4())
    conn.execute("""
        INSERT INTO review_events
            (event_id, task_id, object_id, reviewer_id, decision, rationale,
             rejection_reason, prior_hash, resulting_hash, corrected_artifact_id, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        event_id, task_id, object_id, reviewer_id, decision, rationale.strip(),
        rejection_reason, prior_hash, resulting_hash, corrected_artifact_id, _utc_now(),
    ))
    return event_id


def approve_document_text_and_tables(doc_id: str, reviewer_id: str, rationale: str) -> None:
    """
    Approve active prose & table chunks for a document with a verified publication_date.
    Quarantined figures are released to 'pending_review' (NEVER auto-approved).
    """
    rows = conn.execute("""
        SELECT chunk_id, element_type, review_status, chunk_content_hash, metadata_json
        FROM child_chunks
        WHERE doc_id = ?
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (doc_id, CORPUS_RELEASE)).fetchall()

    if not rows:
        raise ValueError(f"No active chunks found for doc_id={doc_id!r} in release {CORPUS_RELEASE!r}.")

    first_meta = json.loads(rows[0][4])
    if not first_meta.get("publication_date"):
        raise ValueError(f"Cannot approve {doc_id!r}: verified publication_date is missing (remains quarantined).")

    doc_hash = first_meta.get("content_hash", _sha256_text(doc_id))
    task_id = _ensure_review_task(doc_id, "document_text_tables", doc_hash, "pending_review")
    now = _utc_now()

    approved_n = 0
    released_figs = 0

    for cid, etype, rstatus, chash, mjson in rows:
        meta = json.loads(mjson)
        if etype in ("prose", "table"):
            if rstatus != "approved":
                meta["review_status"] = "approved"
                meta["reviewed_at"] = now
                conn.execute("""
                    UPDATE child_chunks
                    SET review_status = 'approved', reviewed_at = ?, metadata_json = ?
                    WHERE chunk_id = ?
                """, (now, json.dumps(meta), cid))
                approved_n += 1
        elif etype == "figure":
            # Never auto-approve figures; only release from quarantine to pending_review
            if rstatus == "quarantined":
                meta["review_status"] = "pending_review"
                meta["reviewed_at"] = None
                conn.execute("""
                    UPDATE child_chunks
                    SET review_status = 'pending_review', reviewed_at = NULL, metadata_json = ?
                    WHERE chunk_id = ?
                """, (json.dumps(meta), cid))
                _ensure_review_task(cid, "figure", chash, "pending_review")
                released_figs += 1
            elif rstatus == "pending_review":
                _ensure_review_task(cid, "figure", chash, "pending_review")

    # Approve parent_chunks for this document (retrieval still reconstructs from approved children only)
    conn.execute("""
        UPDATE parent_chunks
        SET review_status = 'approved'
        WHERE doc_id = ?
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (doc_id, CORPUS_RELEASE))

    conn.execute("UPDATE review_tasks SET status = 'approved', updated_at = ? WHERE task_id = ?", (now, task_id))
    _append_review_event(
        task_id=task_id,
        object_id=doc_id,
        reviewer_id=reviewer_id,
        decision="approve",
        rationale=rationale,
        prior_hash=doc_hash,
        resulting_hash=doc_hash,
    )
    conn.commit()
    print(f"[{doc_id}] Approved {approved_n} prose/table chunk(s); released {released_figs} figure(s) to pending_review.")


def record_figure_decision(
    chunk_id: str,
    reviewer_id: str,
    decision: str,                 # 'approve' | 'reject' | 'correct'
    rationale: str,
    corrected_text: str | None = None,
    rejection_reason: str | None = None,
) -> None:
    """
    Auditable figure state transition:
      pending_review | corrected_pending_review -> approved | rejected | corrected_pending_review
    Corrections create a new derived_artifact with a new hash and MUST be reviewed again.
    """
    if decision not in ("approve", "reject", "correct"):
        raise ValueError(f"Unsupported decision {decision!r}; use 'approve', 'reject', or 'correct'.")

    row = conn.execute("""
        SELECT doc_id, element_type, review_status, chunk_content_hash, text, metadata_json
        FROM child_chunks
        WHERE chunk_id = ?
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (chunk_id, CORPUS_RELEASE)).fetchone()

    if not row:
        raise KeyError(f"Active chunk not found in release {CORPUS_RELEASE!r}: {chunk_id}")
    doc_id, etype, rstatus, cur_hash, cur_text, mjson = row
    if etype != "figure":
        raise ValueError(f"Chunk {chunk_id} is {etype!r}, not 'figure'.")
    if rstatus == "quarantined":
        raise ValueError(f"Cannot review quarantined figure {chunk_id}; verify parent doc publication_date first.")

    meta = json.loads(mjson)
    task_id = _ensure_review_task(chunk_id, "figure", cur_hash, rstatus)
    now = _utc_now()

    if decision == "approve":
        meta["review_status"] = "approved"
        meta["reviewed_at"] = now
        conn.execute("""
            UPDATE child_chunks
            SET review_status = 'approved', reviewed_at = ?, metadata_json = ?
            WHERE chunk_id = ?
        """, (now, json.dumps(meta), chunk_id))
        conn.execute("UPDATE review_tasks SET status = 'approved', updated_at = ? WHERE task_id = ?", (now, task_id))
        _append_review_event(
            task_id=task_id, object_id=chunk_id, reviewer_id=reviewer_id,
            decision="approve", rationale=rationale,
            prior_hash=cur_hash, resulting_hash=cur_hash,
        )

    elif decision == "reject":
        if not rejection_reason:
            raise ValueError("rejection_reason is required when decision='reject'.")
        meta["review_status"] = "rejected"
        meta["reviewed_at"] = None
        conn.execute("""
            UPDATE child_chunks
            SET review_status = 'rejected', reviewed_at = NULL, metadata_json = ?
            WHERE chunk_id = ?
        """, (json.dumps(meta), chunk_id))
        conn.execute("UPDATE review_tasks SET status = 'rejected', updated_at = ? WHERE task_id = ?", (now, task_id))
        _append_review_event(
            task_id=task_id, object_id=chunk_id, reviewer_id=reviewer_id,
            decision="reject", rationale=rationale, rejection_reason=rejection_reason,
            prior_hash=cur_hash, resulting_hash=cur_hash,
        )

    elif decision == "correct":
        if not corrected_text or not corrected_text.strip():
            raise ValueError("corrected_text is required when decision='correct'.")
        new_hash = _sha256_text(corrected_text.strip())
        art_id = str(uuid.uuid4())
        # Store correction as a separate derived artifact; never overwrite raw VLM file on disk
        conn.execute("""
            INSERT INTO derived_artifacts
                (artifact_id, source_chunk_id, prior_hash, corrected_text, resulting_hash, created_by, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (art_id, chunk_id, cur_hash, corrected_text.strip(), new_hash, reviewer_id, now))

        meta["review_status"] = "corrected_pending_review"
        meta["reviewed_at"] = None
        meta["derived_artifact_id"] = art_id
        conn.execute("""
            UPDATE child_chunks
            SET text = ?, chunk_content_hash = ?, review_status = 'corrected_pending_review',
                reviewed_at = NULL, metadata_json = ?
            WHERE chunk_id = ?
        """, (corrected_text.strip(), new_hash, json.dumps(meta), chunk_id))

        # Close old hash task and open a new task bound to the corrected hash
        conn.execute("UPDATE review_tasks SET status = 'corrected_pending_review', updated_at = ? WHERE task_id = ?", (now, task_id))
        _ensure_review_task(chunk_id, "figure", new_hash, "corrected_pending_review")
        _append_review_event(
            task_id=task_id, object_id=chunk_id, reviewer_id=reviewer_id,
            decision="correct", rationale=rationale,
            prior_hash=cur_hash, resulting_hash=new_hash, corrected_artifact_id=art_id,
        )

    conn.commit()
    print(f"[figure {chunk_id[:8]}...] Recorded decision='{decision}' by {reviewer_id}.")


def list_pending_figures(limit: int = 20):
    """Display the active figure review queue."""
    rows = conn.execute("""
        SELECT chunk_id, doc_id, page_start, review_status, section_path,
               json_extract(metadata_json, '$.artifact_path'), substr(text, 1, 140)
        FROM child_chunks
        WHERE element_type = 'figure'
          AND review_status IN ('pending_review', 'corrected_pending_review')
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
        ORDER BY doc_id, page_start, rowid
        LIMIT ?
    """, (CORPUS_RELEASE, limit)).fetchall()

    print(f"=== Pending Figure Review Queue ({len(rows)} shown) ===")
    for cid, did, pg, st, spath, apath, preview in rows:
        print(f"  {cid} | {did} | p.{pg} | {st}")
        print(f"    section : {spath}")
        print(f"    artifact: {apath}")
        print(f"    text    : {preview.replace(chr(10), ' ')}...")
    return rows


def inspect_figure(chunk_id: str) -> None:
    """Side-by-side notebook inspector: original figure crop + current text + audit history."""
    row = conn.execute("""
        SELECT doc_id, page_start, page_end, section_path, review_status,
               chunk_content_hash, text, json_extract(metadata_json, '$.artifact_path')
        FROM child_chunks WHERE chunk_id = ?
    """, (chunk_id,)).fetchone()
    if not row:
        raise KeyError(f"Chunk not found: {chunk_id}")

    did, ps, pe, spath, st, chash, txt, apath = row
    print(f"Chunk ID : {chunk_id}")
    print(f"Source   : {did} | p.{ps}-{pe} | status={st} | hash={chash[:12]}...")
    print(f"Section  : {spath}")
    print("-" * 80)
    print(txt)
    print("-" * 80)

    if apath and Path(apath).exists():
        try:
            from IPython.display import Image as IPyImage, display
            display(IPyImage(filename=apath, width=650))
        except Exception:
            print(f"[Image file on disk: {apath}]")

    events = conn.execute("""
        SELECT created_at, reviewer_id, decision, rationale, rejection_reason
        FROM review_events WHERE object_id = ? ORDER BY created_at
    """, (chunk_id,)).fetchall()
    if events:
        print("Audit History:")
        for ts, rev, dec, rat, rej in events:
            extra = f" | rejection={rej}" if rej else ""
            print(f"  {ts} | {rev} | {dec} | {rat}{extra}")


# ── Example: sync & inspect current review queue ──────────────────────────────
# Uncomment and run with your real reviewer ID when approving verified docs:
# approve_document_text_and_tables("who_arboviral_guideline_2025", reviewer_id="ayan_c", rationale="Verified WHO 2025 publication date and text/table extraction.")
# approve_document_text_and_tables("who_dengue_lab_guidance_2025", reviewer_id="ayan_c", rationale="Verified WHO 2025 lab guidance publication date and text/table extraction.")

_ = list_pending_figures(limit=10)

In [ ]:
import hashlib
from datetime import datetime, timezone
from qdrant_client import models

# Provision release_manifests table in SQLite for auditable release promotion & rollback
conn.executescript("""
CREATE TABLE IF NOT EXISTS release_manifests (
    release_id       TEXT PRIMARY KEY,
    collection_name  TEXT NOT NULL,
    manifest_sha256  TEXT NOT NULL,
    sqlite_count     INTEGER NOT NULL,
    qdrant_count     INTEGER NOT NULL,
    policy_count     INTEGER NOT NULL,
    promoted_by      TEXT,
    promoted_at      TEXT,
    status           TEXT NOT NULL   -- 'staged' | 'active' | 'superseded' | 'failed'
);
""")
conn.commit()

# Scope collection to CORPUS_RELEASE so new releases build in isolation
COLLECTION_NAME = f"guarddengue_{CORPUS_RELEASE}"
DENSE_DIM = len(next(dense_embedder.embed(["dimension probe"])))


def ensure_qdrant_collection(collection_name: str, vector_size: int) -> None:
    if not qdrant.collection_exists(collection_name):
        qdrant.create_collection(
            collection_name=collection_name,
            vectors_config={
                "dense": models.VectorParams(
                    size=vector_size,
                    distance=models.Distance.COSINE,
                )
            },
            sparse_vectors_config={
                "sparse": models.SparseVectorParams(
                    modifier=models.Modifier.IDF,
                )
            },
        )


def _compute_manifest_hash(pairs: list[tuple[str, str]], release_id: str) -> str:
    canonical = json.dumps(
        {"release_id": release_id, "chunks": sorted(pairs)},
        sort_keys=True,
    )
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


# 1. Fetch active, approved child chunks for the target CORPUS_RELEASE
active_rows = conn.execute("""
    SELECT chunk_id, parent_id, doc_id, section_path, page_start, page_end,
           element_type, review_status, reviewed_at, chunk_content_hash,
           text, metadata_json
    FROM child_chunks
    WHERE review_status = 'approved'
      AND superseded_by IS NULL
      AND json_extract(metadata_json, '$.corpus_release') = ?
    ORDER BY doc_id, page_start, rowid
""", (CORPUS_RELEASE,)).fetchall()

expected_pairs = sorted((r[0], r[9]) for r in active_rows)
expected_ids = {cid for cid, _ in expected_pairs}
manifest_hash = _compute_manifest_hash(expected_pairs, CORPUS_RELEASE)

ensure_qdrant_collection(COLLECTION_NAME, DENSE_DIM)

# 2. Batch-embed & upsert active approved chunks into the release collection
BATCH_SIZE = 32
upserted_count = 0

for start in range(0, len(active_rows), BATCH_SIZE):
    batch = active_rows[start : start + BATCH_SIZE]
    texts = [r[10] for r in batch]

    dense_vecs = [v.tolist() for v in dense_embedder.embed(texts)]
    sparse_vecs = list(sparse_embedder.embed(texts))

    points = []
    for row, d_vec, s_vec in zip(batch, dense_vecs, sparse_vecs):
        cid, pid, did, spath, ps, pe, etype, rstatus, rev_at, chash, txt, mjson = row
        meta = json.loads(mjson)
        payload = {
            "chunk_id": cid,
            "parent_id": pid,
            "doc_id": did,
            "source": meta.get("source"),
            "document_version": meta.get("document_version"),
            "publication_date": meta.get("publication_date"),
            "content_hash": meta.get("content_hash"),
            "chunk_content_hash": chash,
            "corpus_release": CORPUS_RELEASE,
            "section_path": spath,
            "page_start": ps,
            "page_end": pe,
            "element_type": etype,
            "review_status": rstatus,
            "reviewed_at": rev_at,
            "superseded_by": None,
            "artifact_path": meta.get("artifact_path"),
            "text": txt,
        }
        points.append(
            models.PointStruct(
                id=cid,
                vector={
                    "dense": d_vec,
                    "sparse": models.SparseVector(
                        indices=s_vec.indices.tolist(),
                        values=s_vec.values.tolist(),
                    ),
                },
                payload=payload,
            )
        )

    qdrant.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    upserted_count += len(points)

# 3. Prune any Qdrant point IDs in this collection that are no longer active in SQLite
existing_ids = set()
offset = None
while True:
    scroll_pts, next_offset = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        limit=256,
        offset=offset,
        with_payload=False,
        with_vectors=False,
    )
    for pt in scroll_pts:
        existing_ids.add(str(pt.id))
    if next_offset is None:
        break

stale_ids = sorted(existing_ids - expected_ids)
if stale_ids:
    for i in range(0, len(stale_ids), 256):
        qdrant.delete(
            collection_name=COLLECTION_NAME,
            points_selector=models.PointIdsList(points=stale_ids[i : i + 256]),
            wait=True,
        )
    print(f"Pruned {len(stale_ids)} stale Qdrant point(s) from {COLLECTION_NAME}.")

# 4. Verify exact ID & count parity before recording manifest
final_qdrant_count = qdrant.get_collection(COLLECTION_NAME).points_count
if final_qdrant_count != len(expected_ids):
    conn.execute("""
        INSERT OR REPLACE INTO release_manifests
            (release_id, collection_name, manifest_sha256, sqlite_count, qdrant_count, policy_count, promoted_by, promoted_at, status)
        VALUES (?, ?, ?, ?, ?, 0, NULL, NULL, 'failed')
    """, (CORPUS_RELEASE, COLLECTION_NAME, manifest_hash, len(expected_ids), final_qdrant_count))
    conn.commit()
    raise RuntimeError(
        f"Release {CORPUS_RELEASE} parity failed: Qdrant={final_qdrant_count} != SQLite={len(expected_ids)}"
    )

policy_rules = (
    json.loads(POLICY_ARTIFACT_PATH.read_text(encoding="utf-8"))
    if "POLICY_ARTIFACT_PATH" in globals() and POLICY_ARTIFACT_PATH.exists()
    else []
)
now_iso = datetime.now(timezone.utc).isoformat()
publisher_id = globals().get("CLINICAL_REVIEWER_ID") or "pipeline_release_runner"

conn.execute("UPDATE release_manifests SET status = 'superseded' WHERE status = 'active'")
conn.execute("""
    INSERT OR REPLACE INTO release_manifests
        (release_id, collection_name, manifest_sha256, sqlite_count, qdrant_count, policy_count, promoted_by, promoted_at, status)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, 'active')
""", (
    CORPUS_RELEASE, COLLECTION_NAME, manifest_hash,
    len(expected_ids), final_qdrant_count, len(policy_rules),
    publisher_id, now_iso,
))
conn.commit()

print(
    f"Promoted release '{CORPUS_RELEASE}' -> collection='{COLLECTION_NAME}' "
    f"({final_qdrant_count} points, manifest={manifest_hash[:12]}...)."
)

In [ ]:
def check_release_sync(verbose: bool = True) -> bool:
    """
    Verify exact ID-set and content-hash parity across SQLite, Qdrant,
    release_manifests, and local policy JSON (plus Neo4j if configured).
    """
    sqlite_rows = conn.execute("""
        SELECT chunk_id, chunk_content_hash
        FROM child_chunks
        WHERE review_status = 'approved'
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (CORPUS_RELEASE,)).fetchall()
    sqlite_map = {cid: chash for cid, chash in sqlite_rows}
    sqlite_ids = set(sqlite_map.keys())

    if not qdrant.collection_exists(COLLECTION_NAME):
        if verbose:
            print(f"[SYNC FAIL] Qdrant collection missing: {COLLECTION_NAME}")
        return False

    qdrant_map = {}
    offset = None
    while True:
        pts, next_offset = qdrant.scroll(
            collection_name=COLLECTION_NAME,
            limit=256,
            offset=offset,
            with_payload=["corpus_release", "review_status", "superseded_by", "chunk_content_hash"],
            with_vectors=False,
        )
        for pt in pts:
            pl = pt.payload or {}
            if (
                pl.get("corpus_release") != CORPUS_RELEASE
                or pl.get("review_status") != "approved"
                or pl.get("superseded_by") is not None
            ):
                if verbose:
                    print(f"[SYNC FAIL] Invalid Qdrant payload on point {pt.id}: {pl}")
                return False
            qdrant_map[str(pt.id)] = pl.get("chunk_content_hash")
        if next_offset is None:
            break

    qdrant_ids = set(qdrant_map.keys())
    if sqlite_ids != qdrant_ids:
        if verbose:
            print(
                f"[SYNC FAIL] ID mismatch: only_sqlite={len(sqlite_ids - qdrant_ids)}, "
                f"only_qdrant={len(qdrant_ids - sqlite_ids)}"
            )
        return False

    hash_mismatches = [cid for cid in sqlite_ids if sqlite_map[cid] != qdrant_map.get(cid)]
    if hash_mismatches:
        if verbose:
            print(f"[SYNC FAIL] Chunk hash mismatch on {len(hash_mismatches)} point(s): {hash_mismatches[:5]}")
        return False

    manifest_row = conn.execute("""
        SELECT manifest_sha256, sqlite_count, qdrant_count, status
        FROM release_manifests
        WHERE release_id = ? AND collection_name = ?
    """, (CORPUS_RELEASE, COLLECTION_NAME)).fetchone()
    if not manifest_row or manifest_row[3] != "active":
        if verbose:
            print(f"[SYNC FAIL] No active release_manifest row for {CORPUS_RELEASE} ({COLLECTION_NAME})")
        return False

    expected_manifest = _compute_manifest_hash(sorted(sqlite_map.items()), CORPUS_RELEASE)
    if manifest_row[0] != expected_manifest:
        if verbose:
            print("[SYNC FAIL] Release manifest hash drift detected.")
        return False

    if "POLICY_ARTIFACT_PATH" in globals() and POLICY_ARTIFACT_PATH.exists():
        rules = json.loads(POLICY_ARTIFACT_PATH.read_text(encoding="utf-8"))
        for r in rules:
            if r.get("corpus_release") != CORPUS_RELEASE:
                if verbose:
                    print(f"[SYNC FAIL] Policy rule {r.get('assertion_id')} release mismatch.")
                return False
            prov = r.get("provenance", {})
            cid = prov.get("chunk_id")
            if cid not in sqlite_map:
                if verbose:
                    print(f"[SYNC FAIL] Policy rule {r.get('assertion_id')} references non-active chunk {cid}.")
                return False
            if prov.get("chunk_content_hash") and prov.get("chunk_content_hash") != sqlite_map[cid]:
                if verbose:
                    print(f"[SYNC FAIL] Policy rule {r.get('assertion_id')} chunk hash drift.")
                return False

    if NEO4J_PASSWORD:
        try:
            from neo4j import GraphDatabase
            with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
                recs, _, _ = driver.execute_query(
                    """
                    MATCH ()-[r:AVOID_IN]->()
                    WHERE r.corpus_release <> $rel OR r.corpus_release IS NULL
                    RETURN count(r) AS bad
                    """,
                    rel=CORPUS_RELEASE,
                    database_=NEO4J_DATABASE,
                    routing_="r",
                )
                if recs and recs[0]["bad"] > 0:
                    if verbose:
                        print(f"[SYNC FAIL] Neo4j has {recs[0]['bad']} stale AVOID_IN edge(s).")
                    return False
        except Exception as exc:
            if verbose:
                print(f"[SYNC FAIL] Neo4j check error: {exc}")
            return False

    if verbose:
        print(f"[SYNC OK] Release '{CORPUS_RELEASE}' verified across SQLite, Qdrant, manifest, and policy.")
    return True

In [ ]:
from datetime import datetime, timezone

def find_approved_chunks(keyword: str, doc_id: str | None = None, limit: int = 5):
    """Helper to inspect active approved chunks in the current release before pinning an assertion."""
    sql = """
        SELECT chunk_id, doc_id, page_start, section_path, substr(text, 1, 200)
        FROM child_chunks
        WHERE review_status = 'approved'
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
          AND lower(text) LIKE ?
    """
    params = [CORPUS_RELEASE, f"%{keyword.lower()}%"]
    if doc_id:
        sql += " AND doc_id = ?"
        params.append(doc_id)
    sql += " ORDER BY doc_id, page_start, rowid LIMIT ?"
    params.append(limit)

    rows = conn.execute(sql, params).fetchall()
    print(f"=== Approved chunks matching '{keyword}' ({len(rows)} shown) ===")
    for cid, did, pg, spath, preview in rows:
        print(f"  {cid} | {did} | p.{pg} | {spath[:55]}")
        print(f"    -> {preview.replace(chr(10), ' ')}...")
    return rows


# 1. Inspect candidate chunks so you can pick and verify the exact dengue recommendation
_ = find_approved_chunks("NSAID") or find_approved_chunks("non-steroidal")

# 2. Explicit clinical pinning — NEVER auto-select nsaid_hits[0][0]
# Paste the verified chunk_id from the printout above and set your reviewer ID.
PINNED_NSAID_CHUNK_ID = None        # e.g. "a1b2c3d4-..."
CLINICAL_REVIEWER_ID = None         # e.g. "dr_name_or_reviewer_id"

# 3. Scoped, typed clinical assertions manifest
REVIEWED_ASSERTIONS_SPEC = [
    {
        "assertion_id": "avoid_nsaids_suspected_dengue_v1",
        "assertion_type": "MEDICATION_CONTRAINDICATION",
        "relation": "AVOID_IN",
        "drug_class": {"id": "nsaids", "name": "NSAIDs"},
        "drug_members": [
            {"id": "aspirin", "name": "Aspirin", "aliases": ["acetylsalicylic acid", "asa", "ecosprin", "disprin"]},
            {"id": "ibuprofen", "name": "Ibuprofen", "aliases": ["brufen", "ibugesic", "advil", "motrin"]},
            {"id": "diclofenac", "name": "Diclofenac", "aliases": ["voveran", "voltaren"]},
            {"id": "naproxen", "name": "Naproxen", "aliases": ["naprosyn", "aleve"]},
            {"id": "mefenamic acid", "name": "Mefenamic acid", "aliases": ["meftal"]},
            {"id": "aceclofenac", "name": "Aceclofenac", "aliases": ["zerodol"]},
            {"id": "nimesulide", "name": "Nimesulide", "aliases": ["nise"]},
        ],
        "clinical_context": {"id": "suspected_dengue", "name": "Suspected dengue"},
        "jurisdiction": "WHO_GLOBAL",
        "population": "all_ages_suspected_dengue",
        "exclusions": [
            "Do not advise stopping an existing prescribed antiplatelet or chronic medication without clinician review."
        ],
        "conflict_status": "none",   # "none" | "unresolved_conflict" | "resolved_by_clinical_owner"
        "rationale": (
            "Avoid NSAIDs such as aspirin and ibuprofen in suspected dengue because of bleeding risk. "
            "Do not advise stopping an existing prescribed medicine without clinician review."
        ),
        "source_chunk_id": PINNED_NSAID_CHUNK_ID,
        "review_status": "approved",
        "reviewed_by": CLINICAL_REVIEWER_ID,
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
    }
]

# 4. Fail-closed validation & provenance binding against active SQLite chunks
REQUIRED_ASSERTION_FIELDS = (
    "assertion_id", "assertion_type", "relation", "jurisdiction",
    "population", "exclusions", "conflict_status", "rationale",
    "source_chunk_id", "review_status", "reviewed_by", "reviewed_at",
)

compiled_assertions = []
for spec in REVIEWED_ASSERTIONS_SPEC:
    missing = [f for f in REQUIRED_ASSERTION_FIELDS if not spec.get(f)]
    if missing:
        raise ValueError(
            f"Fail-closed: assertion {spec.get('assertion_id')!r} missing required fields {missing}. "
            "Pin PINNED_NSAID_CHUNK_ID and set CLINICAL_REVIEWER_ID above before compiling."
        )
    if spec["reviewed_by"] in ("pending_clinical_signoff", "TBD", "prototype"):
        raise ValueError(
            f"Fail-closed: assertion {spec['assertion_id']!r} has placeholder reviewer {spec['reviewed_by']!r}."
        )
    if spec["review_status"] != "approved":
        raise ValueError(
            f"Fail-closed: assertion {spec['assertion_id']!r} is not approved (status={spec['review_status']!r})."
        )

    row = conn.execute("""
        SELECT c.chunk_id, c.parent_id, c.doc_id, c.section_path, c.page_start,
               c.page_end, c.review_status, c.superseded_by, c.chunk_content_hash,
               c.metadata_json
        FROM child_chunks c
        WHERE c.chunk_id = ?
    """, (spec["source_chunk_id"],)).fetchone()

    if not row:
        raise KeyError(f"source_chunk_id not found in SQLite: {spec['source_chunk_id']}")
    if row[6] != "approved":
        raise ValueError(f"Blocked unapproved evidence chunk: {spec['source_chunk_id']} (status={row[6]})")
    if row[7] is not None:
        raise ValueError(
            f"Blocked superseded evidence chunk: {spec['source_chunk_id']} (superseded_by={row[7]})"
        )

    meta = json.loads(row[9])
    if meta.get("corpus_release") != CORPUS_RELEASE:
        raise ValueError(
            f"Blocked release-mismatched chunk: {spec['source_chunk_id']} "
            f"(chunk={meta.get('corpus_release')!r} != {CORPUS_RELEASE!r})"
        )
    if not meta.get("publication_date"):
        raise ValueError(f"Blocked chunk missing publication_date: {spec['source_chunk_id']}")

    compiled_assertions.append({
        **spec,
        "corpus_release": CORPUS_RELEASE,
        "provenance": {
            "doc_id": row[2],
            "source": meta["source"],
            "document_version": meta["document_version"],
            "publication_date": meta["publication_date"],
            "content_hash": meta["content_hash"],
            "chunk_content_hash": row[8],
            "parent_id": row[1],
            "section_path": row[3],
            "chunk_id": row[0],
            "page_start": row[4],
            "page_end": row[5],
        },
    })

if not compiled_assertions:
    raise RuntimeError("No compiled assertions produced — refusing to write empty safety policy artifact.")

# 5. Save versioned deterministic safety policy artifact
POLICY_ARTIFACT_PATH = EXTRACTED_DIR / f"safety_policy_{CORPUS_RELEASE}.json"
POLICY_ARTIFACT_PATH.write_text(json.dumps(compiled_assertions, indent=2), encoding="utf-8")
print(f"\nCompiled {len(compiled_assertions)} signed-off assertion(s) -> {POLICY_ARTIFACT_PATH}")

check_release_sync()

In [ ]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

SCHEMA_CONSTRAINTS = [
    "CREATE CONSTRAINT doc_id_unique IF NOT EXISTS FOR (d:Document) REQUIRE d.doc_id IS UNIQUE",
    "CREATE CONSTRAINT sec_id_unique IF NOT EXISTS FOR (s:Section) REQUIRE s.section_id IS UNIQUE",
    "CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS FOR (c:EvidenceChunk) REQUIRE c.chunk_id IS UNIQUE",
    "CREATE CONSTRAINT drug_id_unique IF NOT EXISTS FOR (dr:Drug) REQUIRE dr.id IS UNIQUE",
    "CREATE CONSTRAINT class_id_unique IF NOT EXISTS FOR (cl:DrugClass) REQUIRE cl.id IS UNIQUE",
    "CREATE CONSTRAINT ctx_id_unique IF NOT EXISTS FOR (cx:ClinicalContext) REQUIRE cx.id IS UNIQUE",
]

UPSERT_ASSERTION_CYPHER = """
MERGE (doc:Document {doc_id: $doc_id})
SET doc.source = $source,
    doc.document_version = $document_version,
    doc.publication_date = $publication_date,
    doc.content_hash = $content_hash,
    doc.corpus_release = $corpus_release

MERGE (sec:Section {section_id: $parent_id})
SET sec.section_path = $section_path,
    sec.doc_id = $doc_id
MERGE (sec)-[:FROM_DOCUMENT]->(doc)

MERGE (ch:EvidenceChunk {chunk_id: $chunk_id})
SET ch.doc_id = $doc_id,
    ch.page_start = $page_start,
    ch.page_end = $page_end,
    ch.section_path = $section_path,
    ch.chunk_content_hash = $chunk_content_hash,
    ch.review_status = $review_status,
    ch.corpus_release = $corpus_release
MERGE (ch)-[:PART_OF]->(sec)

MERGE (cls:DrugClass {id: $class_id})
SET cls.name = $class_name

MERGE (ctx:ClinicalContext {id: $context_id})
SET ctx.name = $context_name

MERGE (cls)-[r:AVOID_IN {
    assertion_id: $assertion_id,
    corpus_release: $corpus_release
}]->(ctx)
SET r.context = $context_id,
    r.source_chunk_id = $chunk_id,
    r.review_status = $review_status,
    r.rationale = $rationale,
    r.reviewed_by = $reviewed_by,
    r.reviewed_at = $reviewed_at

MERGE (cls)-[s:SUPPORTED_BY {
    assertion_id: $assertion_id,
    corpus_release: $corpus_release
}]->(ch)
SET s.review_status = $review_status

WITH cls
UNWIND $drug_members AS dm
MERGE (d:Drug {id: dm.id})
SET d.name = dm.name,
    d.aliases = dm.aliases
MERGE (d)-[:MEMBER_OF]->(cls)
"""

PRUNE_STALE_AVOID_CYPHER = """
MATCH ()-[r:AVOID_IN]->()
WHERE r.corpus_release IS NULL
   OR r.corpus_release <> $corpus_release
   OR NOT r.assertion_id IN $active_assertion_ids
DELETE r
RETURN count(*) AS n
"""

PRUNE_STALE_SUPPORT_CYPHER = """
MATCH ()-[s:SUPPORTED_BY]->(ch:EvidenceChunk)
WHERE s.corpus_release IS NULL
   OR s.corpus_release <> $corpus_release
   OR NOT any(p IN $active_pairs WHERE p.assertion_id = s.assertion_id AND p.chunk_id = ch.chunk_id)
DELETE s
RETURN count(*) AS n
"""

if not NEO4J_PASSWORD:
    print("NEO4J_PASSWORD not set -> skipped live Neo4j write.")
    print(f"Set NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD when Neo4j is running. Local policy file is ready at: {POLICY_ARTIFACT_PATH}")
elif not compiled_assertions:
    print("No compiled assertions to load.")
else:
    from neo4j import GraphDatabase

    # Safety guards before touching Neo4j
    for item in compiled_assertions:
        if item.get("review_status") != "approved":
            raise ValueError(f"Blocked unapproved assertion: {item.get('assertion_id')}")
        if item.get("corpus_release") != CORPUS_RELEASE:
            raise ValueError(
                f"Blocked release-mismatched assertion {item.get('assertion_id')}: "
                f"{item.get('corpus_release')!r} != {CORPUS_RELEASE!r}"
            )
        if item.get("relation") != "AVOID_IN":
            raise ValueError(
                f"Unsupported relation {item.get('relation')!r} for {item.get('assertion_id')}; "
                "add a dedicated Cypher handler before loading."
            )

    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
        driver.verify_connectivity()

        # 1. Ensure unique ID constraints exist
        for stmt in SCHEMA_CONSTRAINTS:
            driver.execute_query(stmt, database_=NEO4J_DATABASE)

        # 2. Upsert each reviewed assertion with full provenance
        for item in compiled_assertions:
            prov = item["provenance"]
            driver.execute_query(
                UPSERT_ASSERTION_CYPHER,
                doc_id=prov["doc_id"],
                source=prov["source"],
                document_version=prov["document_version"],
                publication_date=prov["publication_date"],
                content_hash=prov["content_hash"],
                corpus_release=item["corpus_release"],
                parent_id=prov["parent_id"],
                section_path=prov["section_path"],
                chunk_id=prov["chunk_id"],
                chunk_content_hash=prov.get("chunk_content_hash"),
                page_start=prov["page_start"],
                page_end=prov["page_end"],
                review_status=item["review_status"],
                class_id=item["drug_class"]["id"],
                class_name=item["drug_class"]["name"],
                context_id=item["clinical_context"]["id"],
                context_name=item["clinical_context"]["name"],
                assertion_id=item["assertion_id"],
                rationale=item["rationale"],
                reviewed_by=item["reviewed_by"],
                reviewed_at=item["reviewed_at"],
                drug_members=item["drug_members"],
                database_=NEO4J_DATABASE,
            )

        # 3. Prune stale release / removed / re-pinned edges
        active_ids = [a["assertion_id"] for a in compiled_assertions]
        active_pairs = [
            {"assertion_id": a["assertion_id"], "chunk_id": a["provenance"]["chunk_id"]}
            for a in compiled_assertions
        ]
        recs_a, _, _ = driver.execute_query(
            PRUNE_STALE_AVOID_CYPHER,
            corpus_release=CORPUS_RELEASE,
            active_assertion_ids=active_ids,
            database_=NEO4J_DATABASE,
        )
        recs_s, _, _ = driver.execute_query(
            PRUNE_STALE_SUPPORT_CYPHER,
            corpus_release=CORPUS_RELEASE,
            active_pairs=active_pairs,
            database_=NEO4J_DATABASE,
        )
        da = recs_a[0]["n"] if recs_a else 0
        ds = recs_s[0]["n"] if recs_s else 0
        if da or ds:
            print(f"Pruned {da} stale AVOID_IN and {ds} stale SUPPORTED_BY edge(s).")

    print(f"Loaded {len(compiled_assertions)} reviewed assertion(s) into Neo4j ({NEO4J_URI}, db={NEO4J_DATABASE}).")
    check_release_sync()

In [ ]:
import re

# Build trusted allowlist from active approved child chunks for the current release
APPROVED_SOURCES = sorted({
    json.loads(mjson)["source"]
    for (mjson,) in conn.execute("""
        SELECT metadata_json FROM child_chunks
        WHERE review_status = 'approved'
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (CORPUS_RELEASE,)).fetchall()
})

# Deterministic Indian brand, alias, and Fixed-Dose Combination (FDC) -> ingredient map.
# Expand this table from a clinically reviewed formulary before production.
DRUG_NORMALIZATION_MAP = {
    # Single-ingredient NSAIDs & common Indian brands
    "aspirin": ["aspirin"],
    "acetylsalicylic acid": ["aspirin"],
    "asa": ["aspirin"],
    "ecosprin": ["aspirin"],
    "disprin": ["aspirin"],
    "ibuprofen": ["ibuprofen"],
    "brufen": ["ibuprofen"],
    "ibugesic": ["ibuprofen"],
    "advil": ["ibuprofen"],
    "motrin": ["ibuprofen"],
    "diclofenac": ["diclofenac"],
    "voveran": ["diclofenac"],
    "voltaren": ["diclofenac"],
    "naproxen": ["naproxen"],
    "naprosyn": ["naproxen"],
    "aleve": ["naproxen"],
    "mefenamic acid": ["mefenamic acid"],
    "meftal": ["mefenamic acid"],
    "aceclofenac": ["aceclofenac"],
    "zerodol": ["aceclofenac"],
    "nimesulide": ["nimesulide"],
    "nise": ["nimesulide"],
    # Fixed-Dose Combinations (FDCs) common in India — must expand to ALL ingredients
    "combiflam": ["ibuprofen", "paracetamol"],
    "flexon": ["ibuprofen", "paracetamol"],
    "ibugesic plus": ["ibuprofen", "paracetamol"],
    "meftal spas": ["mefenamic acid", "dicyclomine"],
    "zerodol p": ["aceclofenac", "paracetamol"],
    "sumo": ["nimesulide", "paracetamol"],
    # Non-NSAID antipyretic (tracked so detector does not return 'unresolved')
    "paracetamol": ["paracetamol"],
    "acetaminophen": ["paracetamol"],
    "dolo": ["paracetamol"],
    "crocin": ["paracetamol"],
    "calpol": ["paracetamol"],
}

_PRESCRIBED_RE = re.compile(
    r"\b(prescribed|regularly|daily|doctor\s+(?:prescribed|gave|told)|already\s+taking|on\s+low[- ]dose)\b",
    re.IGNORECASE,
)
_NEGATION_RE = re.compile(
    r"\b(not\s+taking|avoid|don't\s+take|do\s+not\s+take|without|stopped)\b",
    re.IGNORECASE,
)


def extract_and_normalize_drugs(
    query_text: str,
    extra_terms: list[str] | None = None,
) -> dict:
    """
    Independent deterministic medication-mention detector and ingredient normalizer.
    Expands Indian brands and FDCs to canonical ingredients and flags prescribed/negated context.
    """
    q_lower = (query_text or "").lower()
    detected_mentions = []
    normalized_ingredients = []
    unresolved_terms = []

    # Match longest aliases first so 'ibugesic plus' beats 'ibugesic'
    for alias in sorted(DRUG_NORMALIZATION_MAP.keys(), key=len, reverse=True):
        if re.search(rf"\b{re.escape(alias)}\b", q_lower):
            detected_mentions.append(alias)
            for ing in DRUG_NORMALIZATION_MAP[alias]:
                if ing not in normalized_ingredients:
                    normalized_ingredients.append(ing)

    # Also normalize any caller-supplied terms; track unknown ones as unresolved
    for raw in (extra_terms or []):
        clean = (raw or "").strip().lower()
        if not clean:
            continue
        if clean in DRUG_NORMALIZATION_MAP:
            if clean not in detected_mentions:
                detected_mentions.append(clean)
            for ing in DRUG_NORMALIZATION_MAP[clean]:
                if ing not in normalized_ingredients:
                    normalized_ingredients.append(ing)
        else:
            if clean not in unresolved_terms:
                unresolved_terms.append(clean)

    return {
        "detected_mentions": detected_mentions,
        "normalized_ingredients": normalized_ingredients,
        "unresolved_terms": unresolved_terms,
        "mentions_existing_prescription": bool(_PRESCRIBED_RE.search(q_lower)),
        "has_negation_cue": bool(_NEGATION_RE.search(q_lower)),
    }


def get_jurisdiction_coverage() -> dict:
    """Report which guideline authorities are active vs. quarantined in SQLite."""
    rows = conn.execute("""
        SELECT doc_id, review_status, COUNT(*)
        FROM child_chunks
        WHERE superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
        GROUP BY doc_id, review_status
    """, (CORPUS_RELEASE,)).fetchall()

    doc_statuses: dict[str, set[str]] = {}
    for doc_id, status, _ in rows:
        doc_statuses.setdefault(doc_id, set()).add(status)

    coverage = {}
    for doc_id, statuses in sorted(doc_statuses.items()):
        if "approved" in statuses and "quarantined" not in statuses:
            coverage[doc_id] = "available" if "pending_review" not in statuses else "partial_figures_pending"
        elif "quarantined" in statuses:
            coverage[doc_id] = "unavailable_quarantined"
        else:
            coverage[doc_id] = "unavailable_pending_review"
    return coverage


def hybrid_retrieve(query_text: str, limit: int = 5, max_parent_chars: int = 2000) -> dict:
    """
    Filtered dense+sparse hybrid retrieval with explicit completeness reporting.
    - Never slices mid-unit (replaces blind [:2000] truncation with whole typed units).
    - Reports unresolved (pending_review / quarantined) figures & tables per retrieved section.
    - Reports per-document / jurisdiction availability.
    """
    if not APPROVED_SOURCES or not CORPUS_RELEASE:
        raise ValueError("Approved source allowlist and corpus_release are required.")

    q_dense = next(dense_embedder.embed([query_text])).tolist()
    q_sparse_raw = next(sparse_embedder.query_embed([query_text]))
    q_sparse = models.SparseVector(
        indices=q_sparse_raw.indices.tolist(),
        values=q_sparse_raw.values.tolist(),
    )

    eligible_filter = models.Filter(must=[
        models.FieldCondition(key="source",         match=models.MatchAny(any=APPROVED_SOURCES)),
        models.FieldCondition(key="review_status",  match=models.MatchValue(value="approved")),
        models.FieldCondition(key="corpus_release", match=models.MatchValue(value=CORPUS_RELEASE)),
        models.IsEmptyCondition(is_empty=models.PayloadField(key="superseded_by")),
    ])

    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=q_dense, using="dense", filter=eligible_filter, limit=30),
            models.Prefetch(query=q_sparse, using="sparse", filter=eligible_filter, limit=30),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        query_filter=eligible_filter,
        limit=limit,
        with_payload=True,
    ).points

    verified_results = []
    all_unresolved_objects = []
    seen_unresolved = set()
    any_units_omitted = False

    for hit in hits:
        p = hit.payload or {}

        # Explicit runtime guards (never stripped by python -O)
        if p.get("review_status") != "approved":
            raise RuntimeError(f"Unapproved chunk leaked: {p.get('chunk_id')}")
        if p.get("corpus_release") != CORPUS_RELEASE:
            raise RuntimeError(f"Release mismatch: {p.get('corpus_release')}")
        if p.get("superseded_by") is not None:
            raise RuntimeError(f"Superseded chunk leaked: {p.get('chunk_id')}")
        if p.get("source") not in APPROVED_SOURCES:
            raise RuntimeError(f"Untrusted source leaked: {p.get('source')}")
        if not (p.get("publication_date") and p.get("section_path") and p.get("chunk_id") and p.get("parent_id")):
            raise RuntimeError(f"Incomplete provenance on chunk: {p.get('chunk_id')}")

        parent_row = conn.execute(
            "SELECT section_path, page_start, page_end, review_status FROM parent_chunks WHERE parent_id=?",
            (p["parent_id"],),
        ).fetchone()
        if parent_row is None or parent_row[3] != "approved":
            raise RuntimeError(f"Broken or unapproved parent link: {p['parent_id']}")

        # Fetch ALL active children in this section to separate approved units from unresolved objects
        section_children = conn.execute("""
            SELECT chunk_id, element_type, review_status, page_start, page_end,
                   artifact_path, text
            FROM child_chunks
            WHERE parent_id = ?
              AND superseded_by IS NULL
              AND json_extract(metadata_json, '$.corpus_release') = ?
            ORDER BY rowid
        """, (p["parent_id"], CORPUS_RELEASE)).fetchall()

        approved_units = []
        section_unresolved = []
        for cid, etype, rstatus, ps, pe, apath, txt in section_children:
            if rstatus == "approved":
                approved_units.append({
                    "chunk_id": cid,
                    "element_type": etype,
                    "page_start": ps,
                    "page_end": pe,
                    "artifact_path": apath,
                    "text": txt,
                })
            else:
                obj = {
                    "chunk_id": cid,
                    "parent_id": p["parent_id"],
                    "doc_id": p["doc_id"],
                    "element_type": etype,
                    "review_status": rstatus,
                    "section_path": parent_row[0],
                    "page": f"{ps}-{pe}",
                    "artifact_path": apath,
                }
                section_unresolved.append(obj)
                if cid not in seen_unresolved:
                    seen_unresolved.add(cid)
                    all_unresolved_objects.append(obj)

        if not approved_units:
            raise RuntimeError(f"No active approved sibling chunks found for parent: {p['parent_id']}")

        # Qualifier-preserving unit assembly: always include the matched hit intact,
        # then add whole sibling units only if they fit without mid-unit slicing.
        matched_unit = next((u for u in approved_units if u["chunk_id"] == p["chunk_id"]), approved_units[0])
        selected_units = [matched_unit]
        omitted_unit_ids = []
        used_chars = len(matched_unit["text"])

        for u in approved_units:
            if u["chunk_id"] == matched_unit["chunk_id"]:
                continue
            if used_chars + len(u["text"]) <= max_parent_chars:
                selected_units.append(u)
                used_chars += len(u["text"])
            else:
                omitted_unit_ids.append(u["chunk_id"])

        if omitted_unit_ids:
            any_units_omitted = True

        art_path = p.get("artifact_path")
        art_exists = Path(art_path).exists() if art_path else None

        verified_results.append({
            "score":               round(hit.score, 4),
            "doc_id":              p["doc_id"],
            "source":              p["source"],
            "publication_date":    p["publication_date"],
            "element_type":        p["element_type"],
            "page":                f"{p.get('page_start')}-{p.get('page_end')}",
            "section_path":        p["section_path"],
            "chunk_id":            p["chunk_id"],
            "parent_id":           p["parent_id"],
            "parent_pages":        f"{parent_row[1]}-{parent_row[2]}",
            "artifact_path":       art_path,
            "artifact_exists":     art_exists,
            "child_snippet":       (p.get("text") or "")[:220].replace("\n", " "),
            "context_units":       selected_units,      # intact typed units; never mid-sliced
            "truncation_status":   "units_omitted" if omitted_unit_ids else "none",
            "omitted_unit_ids":    omitted_unit_ids,
            "unresolved_in_section": section_unresolved,
        })

    jurisdiction_coverage = get_jurisdiction_coverage()
    has_quarantined_docs = any("quarantined" in v for v in jurisdiction_coverage.values())

    if not verified_results:
        coverage_status = "unavailable"
    elif all_unresolved_objects or any_units_omitted or has_quarantined_docs:
        coverage_status = "partial"
    else:
        coverage_status = "complete"

    return {
        "results":                     verified_results,
        "coverage_status":             coverage_status,
        "unresolved_required_objects": all_unresolved_objects,
        "truncation_occurred":         any_units_omitted,
        "jurisdiction_coverage":       jurisdiction_coverage,
        "corpus_release":              CORPUS_RELEASE,
    }


def retrieve_with_safety_context(
    query_text: str,
    detected_drugs: list[str] | None = None,
    limit: int = 5,
) -> dict:
    """
    Runs completeness-aware hybrid_retrieve(), then independently detects/normalizes
    medication mentions (including Indian brands & FDCs) and queries deterministic policy.
    """
    retrieval = hybrid_retrieve(query_text, limit=limit)
    drug_analysis = extract_and_normalize_drugs(query_text, extra_terms=detected_drugs)

    if drug_analysis["normalized_ingredients"] and "lookup_medication_policy" not in globals():
        raise NameError(
            "lookup_medication_policy() is not defined yet. "
            "Run the medication-policy lookup cell before this cell."
        )

    safety_rules = []
    unmatched_ingredients = []
    seen_rules = set()

    for ingredient in drug_analysis["normalized_ingredients"]:
        result = lookup_medication_policy(ingredient)
        if result.get("status") == "avoid_rule_matched":
            for m in result.get("matches", []):
                key = (m.get("assertion_id"), m.get("drug_name"))
                if key not in seen_rules:
                    seen_rules.add(key)
                    safety_rules.append(m)
        else:
            unmatched_ingredients.append({
                "ingredient": ingredient,
                "status": result.get("status"),
                "safety_note": result.get("safety_note"),
            })

    return {
        "query":                       query_text,
        "corpus_release":              CORPUS_RELEASE,
        "rag_results":                 retrieval["results"],
        "coverage_status":             retrieval["coverage_status"],
        "unresolved_required_objects": retrieval["unresolved_required_objects"],
        "truncation_occurred":         retrieval["truncation_occurred"],
        "jurisdiction_coverage":       retrieval["jurisdiction_coverage"],
        "drug_analysis":               drug_analysis,
        "safety_rules":                safety_rules,
        "unmatched_ingredients":       unmatched_ingredients,
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
test_queries = [
    ("What are the warning signs of severe dengue and can I take Combiflam or Ecosprin?", None),
    ("When should NS1 antigen testing versus IgM serology be used for dengue diagnosis?", None),
]

for q, drugs in test_queries:
    print("=" * 85)
    print(f"QUERY: {q}")
    ctx = retrieve_with_safety_context(q, detected_drugs=drugs, limit=4)
    print(f"Coverage   : {ctx['coverage_status']} | truncation={ctx['truncation_occurred']} | unresolved_objs={len(ctx['unresolved_required_objects'])}")
    print(f"Sources    : {ctx['jurisdiction_coverage']}")
    print(f"RAG hits   : {len(ctx['rag_results'])}")
    for idx, r in enumerate(ctx["rag_results"], 1):
        print(
            f"  [{idx}] score={r['score']} | {r['element_type']} | {r['doc_id']} | p.{r['page']} "
            f"| units={len(r['context_units'])} (omitted={len(r['omitted_unit_ids'])}, unresolved={len(r['unresolved_in_section'])})"
        )
        print(f"       {r['child_snippet'][:150]}...")
    da = ctx["drug_analysis"]
    if da["detected_mentions"] or da["unresolved_terms"]:
        print(f"Drugs      : mentions={da['detected_mentions']} -> ingredients={da['normalized_ingredients']} | unresolved={da['unresolved_terms']}")
    if ctx["safety_rules"]:
        print(f"Safety rules: {len(ctx['safety_rules'])} deterministic rule(s) attached:")
        for sr in ctx["safety_rules"]:
            print(f"  -> {sr.get('drug_name')} ({sr.get('drug_class')}) | {sr.get('rationale', '')[:80]}")
    else:
        print("Safety rules: none triggered.")

In [ ]:
CYPHER_POLICY_LOOKUP = """
MATCH (d:Drug)-[:MEMBER_OF]->(c:DrugClass)
WHERE toLower(d.id) = $term
   OR toLower(d.name) = $term
   OR any(a IN coalesce(d.aliases, []) WHERE toLower(a) = $term)
MATCH (c)-[r:AVOID_IN]->(ctx:ClinicalContext {id: $context})
MATCH (c)-[s:SUPPORTED_BY]->(ch:EvidenceChunk {chunk_id: r.source_chunk_id})
      -[:PART_OF]->(sec:Section)-[:FROM_DOCUMENT]->(doc:Document)
WHERE r.review_status = 'approved'
  AND r.corpus_release = $release
  AND s.assertion_id = r.assertion_id
  AND s.review_status = 'approved'
  AND s.corpus_release = $release
  AND ch.review_status = 'approved'
  AND ch.corpus_release = $release
RETURN r.assertion_id AS assertion_id,
       r.reviewed_by AS reviewed_by,
       d.id AS drug_id,
       d.name AS drug_name,
       c.name AS drug_class,
       ctx.id AS context,
       r.rationale AS rationale,
       doc.doc_id AS doc_id,
       doc.source AS source,
       sec.section_path AS section_path,
       ch.page_start AS page_start,
       ch.page_end AS page_end,
       ch.chunk_id AS chunk_id,
       ch.chunk_content_hash AS chunk_content_hash
"""

_INVALID_REVIEWERS = {None, "", "pending_clinical_signoff", "TBD", "prototype"}


def verify_chunk_in_sqlite(chunk_id: str, expected_hash: str | None = None) -> dict:
    """Confirm policy provenance chunk_id matches an active approved chunk in SQLite."""
    row = conn.execute("""
        SELECT doc_id, section_path, page_start, page_end,
               review_status, superseded_by, chunk_content_hash,
               metadata_json, substr(text, 1, 160)
        FROM child_chunks WHERE chunk_id = ?
    """, (chunk_id,)).fetchone()
    if row is None:
        raise RuntimeError(f"Policy chunk_id not found in SQLite: {chunk_id}")
    if row[4] != "approved":
        raise RuntimeError(f"Policy points to unapproved chunk: {chunk_id} (status={row[4]})")
    if row[5] is not None:
        raise RuntimeError(f"Policy points to superseded chunk: {chunk_id} (superseded_by={row[5]})")

    meta = json.loads(row[7])
    if meta.get("corpus_release") != CORPUS_RELEASE:
        raise RuntimeError(
            f"Policy chunk release mismatch: {meta.get('corpus_release')!r} != {CORPUS_RELEASE!r}"
        )
    if not meta.get("publication_date"):
        raise RuntimeError(f"Policy chunk missing publication_date: {chunk_id}")
    if expected_hash and row[6] and expected_hash != row[6]:
        raise RuntimeError(f"Chunk content hash drift detected for {chunk_id}")

    return {
        "sqlite_verified": True,
        "sqlite_preview": row[8].replace("\n", " "),
    }


def lookup_medication_policy(drug_term: str, context: str = "suspected_dengue") -> dict:
    term = (drug_term or "").strip().lower()
    if not term:
        return {
            "query_term": drug_term,
            "backend": "none",
            "status": "drug_mention_unresolved",
            "safety_note": "Empty or unresolved drug term; cannot establish medication safety.",
            "matches": [],
        }

    matches = []
    backend = "local_policy_fallback"

    # 1. Try Neo4j first if configured; gracefully fall back to versioned local JSON if offline
    if NEO4J_PASSWORD:
        try:
            from neo4j import GraphDatabase
            with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
                records, _, _ = driver.execute_query(
                    CYPHER_POLICY_LOOKUP,
                    term=term,
                    context=context,
                    release=CORPUS_RELEASE,
                    database_=NEO4J_DATABASE,
                    routing_="r",
                )
                matches = [dict(rec) for rec in records]
                backend = "neo4j"
        except Exception as exc:
            print(f"[WARN] Neo4j unavailable ({exc}); falling back to local policy artifact.")

    if backend == "local_policy_fallback":
        if not POLICY_ARTIFACT_PATH.exists():
            return {
                "query_term": drug_term,
                "backend": backend,
                "status": "policy_unavailable",
                "safety_note": f"Policy artifact missing ({POLICY_ARTIFACT_PATH.name}); fail-closed.",
                "matches": [],
            }
        rules = json.loads(POLICY_ARTIFACT_PATH.read_text(encoding="utf-8"))
        for rule in rules:
            if (
                rule.get("review_status") != "approved"
                or rule.get("reviewed_by") in _INVALID_REVIEWERS
                or rule.get("corpus_release") != CORPUS_RELEASE
                or rule.get("relation") != "AVOID_IN"
                or rule["clinical_context"]["id"] != context
            ):
                continue
            for dm in rule["drug_members"]:
                candidates = {
                    dm["id"].lower(),
                    dm["name"].lower(),
                    *(a.lower() for a in dm.get("aliases", [])),
                }
                if term in candidates:
                    prov = rule["provenance"]
                    matches.append({
                        "assertion_id": rule["assertion_id"],
                        "reviewed_by": rule["reviewed_by"],
                        "drug_id": dm["id"],
                        "drug_name": dm["name"],
                        "drug_class": rule["drug_class"]["name"],
                        "context": context,
                        "jurisdiction": rule.get("jurisdiction"),
                        "population": rule.get("population"),
                        "exclusions": rule.get("exclusions", []),
                        "conflict_status": rule.get("conflict_status", "none"),
                        "rationale": rule["rationale"],
                        "doc_id": prov["doc_id"],
                        "source": prov["source"],
                        "section_path": prov["section_path"],
                        "page_start": prov["page_start"],
                        "page_end": prov["page_end"],
                        "chunk_id": prov["chunk_id"],
                        "chunk_content_hash": prov.get("chunk_content_hash"),
                    })

    # 2. Enforce clinical sign-off and cross-verify every matched rule against SQLite
    verified_matches = []
    for m in matches:
        if m.get("reviewed_by") in _INVALID_REVIEWERS:
            raise RuntimeError(
                f"Fail-closed: matched rule {m.get('assertion_id')!r} lacks clinical sign-off."
            )
        m.update(verify_chunk_in_sqlite(m["chunk_id"], expected_hash=m.get("chunk_content_hash")))
        verified_matches.append(m)

    if not verified_matches:
        return {
            "query_term": drug_term,
            "backend": backend,
            "status": "no_reviewed_rule_matched",
            "safety_note": "Absence of a reviewed rule does NOT establish medication safety; do not recommend use.",
            "matches": [],
        }

    if any(m.get("conflict_status") == "unresolved_conflict" for m in verified_matches):
        return {
            "query_term": drug_term,
            "backend": backend,
            "status": "source_conflict",
            "safety_note": "Unresolved guideline conflict detected; apply conservative handling and defer to clinician.",
            "matches": verified_matches,
        }

    return {
        "query_term": drug_term,
        "backend": backend,
        "status": "avoid_rule_matched",
        "matches": verified_matches,
    }


# Test canonical drug name, brand alias, and an unknown medicine
for test_drug in ("aspirin", "brufen", "unknown_medicine_xyz"):
    res = lookup_medication_policy(test_drug)
    print("=" * 85)
    print(f"DRUG CHECK: '{test_drug}' | backend={res['backend']} | status={res['status']}")
    if res["matches"]:
        for m in res["matches"]:
            print(f"  Matched   : {m['drug_name']} -> {m['drug_class']} (context={m['context']}, assertion={m['assertion_id']})")
            print(f"  Scope     : jurisdiction={m.get('jurisdiction')} | pop={m.get('population')} | conflict={m.get('conflict_status')}")
            print(f"  Rationale : {m['rationale']}")
            print(f"  Provenance: doc={m['doc_id']} | p.{m['page_start']}-{m['page_end']} | sec={m['section_path']}")
            print(f"  Chunk ID  : {m['chunk_id']} (SQLite verified={m['sqlite_verified']})")
    else:
        print(f"  Note: {res['safety_note']}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# VERIFICATION CELL — sanity-checks fixes against live stores
# ════════════════════════════════════════════════════════════════════════════
import inspect

results = {}

# ── Check 1: invalid topic / evidence_level are rejected ───────────────────
try:
    validate_registry_entry("test.pdf", {
        "topic": ["Dengue basics", "INVALID_TOPIC"],
        "evidence_level": "ungraded",
        "publication_date": "2025-01-01",
    })
    results["1a_vocab_reject"] = "FAIL  invalid topic was not caught"
except ValueError as e:
    results["1a_vocab_reject"] = f"PASS  ({str(e)[:80]})"

try:
    validate_registry_entry("test.pdf", {
        "topic": ["Dengue basics"],
        "evidence_level": "guideline",
        "publication_date": "2025-01-01",
    })
    results["1b_evidence_reject"] = "FAIL  invalid evidence_level was not caught"
except ValueError as e:
    results["1b_evidence_reject"] = f"PASS  ({str(e)[:80]})"

# ── Check 2: superseded_by filter uses IsEmptyCondition & no mid-slice cap ──
try:
    src = inspect.getsource(hybrid_retrieve)
    if "IsEmptyCondition" in src and "unresolved_in_section" in src:
        results["2_retrieval_completeness"] = "PASS  uses IsEmptyCondition & tracks section completeness"
    else:
        results["2_retrieval_completeness"] = "FAIL  missing IsEmptyCondition or completeness fields"
except Exception as exc:
    results["2_retrieval_completeness"] = f"ERROR {exc}"

# ── Check 3: Qdrant has 0 superseded points & matches SQLite active count ──
try:
    superseded_in_qdrant = qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=models.Filter(must_not=[
            models.IsEmptyCondition(is_empty=models.PayloadField(key="superseded_by"))
        ]),
        exact=True,
    ).count
    expected_active = conn.execute("""
        SELECT COUNT(*) FROM child_chunks
        WHERE review_status = 'approved'
          AND superseded_by IS NULL
          AND json_extract(metadata_json, '$.corpus_release') = ?
    """, (CORPUS_RELEASE,)).fetchone()[0]
    actual_qdrant = qdrant.get_collection(COLLECTION_NAME).points_count

    if superseded_in_qdrant > 0:
        results["3_qdrant_active_parity"] = f"FAIL  {superseded_in_qdrant} superseded point(s) in Qdrant"
    elif actual_qdrant != expected_active:
        results["3_qdrant_active_parity"] = (
            f"FAIL  Qdrant ({actual_qdrant}) != SQLite active approved ({expected_active})"
        )
    else:
        results["3_qdrant_active_parity"] = (
            f"PASS  0 superseded points; Qdrant matches SQLite ({actual_qdrant})"
        )
except Exception as exc:
    results["3_qdrant_active_parity"] = f"ERROR {exc}"

# ── Check 4: all non-approved active figures have reviewed_at = None ───────
try:
    fig_rows = conn.execute("""
        SELECT chunk_id, reviewed_at, json_extract(metadata_json, '$.reviewed_at')
        FROM child_chunks
        WHERE element_type = 'figure'
          AND review_status != 'approved'
          AND superseded_by IS NULL
    """).fetchall()
    bad = [r for r in fig_rows if r[1] is not None or r[2] is not None]
    if bad:
        results["4_fig_reviewed_at"] = (
            f"FAIL  {len(bad)}/{len(fig_rows)} unapproved figure(s) have non-null reviewed_at"
        )
    elif fig_rows:
        results["4_fig_reviewed_at"] = (
            f"PASS  all {len(fig_rows)} unapproved figure chunk(s) have reviewed_at=None"
        )
    else:
        results["4_fig_reviewed_at"] = "SKIP  no unapproved figure chunks in DB"
except Exception as exc:
    results["4_fig_reviewed_at"] = f"ERROR {exc}"

# ── Check 5: list items and table footnotes appear in active SQLite text ───
try:
    list_rows = conn.execute("""
        SELECT text FROM child_chunks
        WHERE element_type = 'prose'
          AND superseded_by IS NULL
          AND text LIKE '%• %'
        LIMIT 1
    """).fetchone()
    results["5a_list_markers"] = (
        f"PASS  sample: {list_rows[0][:80].replace(chr(10), ' ')}..."
        if list_rows
        else "SKIP  no bullet-list chunks found"
    )

    fn_rows = conn.execute("""
        SELECT text FROM child_chunks
        WHERE element_type = 'table'
          AND superseded_by IS NULL
          AND text LIKE '%[footnote]%'
        LIMIT 1
    """).fetchone()
    results["5b_table_footnotes"] = (
        f"PASS  sample: {fn_rows[0][:80].replace(chr(10), ' ')}..."
        if fn_rows
        else "SKIP  no footnote chunks found"
    )
except Exception as exc:
    results["5_list_table"] = f"ERROR {exc}"

# ── Check 6: chunk_ids are non-empty, unique uuid5 strings ─────────────────
try:
    all_ids = [r[0] for r in conn.execute("SELECT chunk_id FROM child_chunks").fetchall()]
    if not all_ids:
        results["6_unique_ids"] = "FAIL  child_chunks table is empty"
    else:
        unique_ids = set(all_ids)
        invalid = []
        for cid in all_ids:
            try:
                if uuid.UUID(cid).version != 5:
                    invalid.append(cid)
            except (ValueError, AttributeError):
                invalid.append(cid)
        if len(all_ids) != len(unique_ids):
            results["6_unique_ids"] = f"FAIL  {len(all_ids) - len(unique_ids)} duplicate chunk_id(s) found"
        elif invalid:
            results["6_unique_ids"] = f"FAIL  {len(invalid)} non-uuid5 chunk_id(s)"
        else:
            results["6_unique_ids"] = f"PASS  {len(all_ids)} unique uuid5 chunk_ids"
except Exception as exc:
    results["6_unique_ids"] = f"ERROR {exc}"

# ── Check 7: quarantined docs & unapproved chunks absent from Qdrant ───────
try:
    unapproved_in_qdrant = qdrant.count(
        collection_name=COLLECTION_NAME,
        count_filter=models.Filter(must_not=[
            models.FieldCondition(key="review_status", match=models.MatchValue(value="approved"))
        ]),
        exact=True,
    ).count
    q_docs = [
        r[0] for r in conn.execute(
            "SELECT DISTINCT doc_id FROM child_chunks WHERE review_status = 'quarantined'"
        ).fetchall()
    ]
    if unapproved_in_qdrant > 0:
        results["7_quarantine_exclusion"] = (
            f"FAIL  {unapproved_in_qdrant} unapproved point(s) leaked into Qdrant"
        )
    elif not q_docs:
        results["7_quarantine_exclusion"] = (
            "PASS  0 unapproved points in Qdrant (no quarantined docs currently in SQLite)"
        )
    else:
        leaked = qdrant.count(
            collection_name=COLLECTION_NAME,
            count_filter=models.Filter(must=[
                models.FieldCondition(key="doc_id", match=models.MatchAny(any=q_docs))
            ]),
            exact=True,
        ).count
        results["7_quarantine_exclusion"] = (
            f"PASS  {len(q_docs)} quarantined doc(s) have 0 points in Qdrant"
            if leaked == 0
            else f"FAIL  quarantined doc(s) {q_docs} have {leaked} point(s) in Qdrant"
        )
except Exception as exc:
    results["7_quarantine_exclusion"] = f"ERROR {exc}"

# ── Check 8: check_release_sync() passes ───────────────────────────────────
try:
    sync_ok = check_release_sync(verbose=False)
    results["8_release_sync"] = (
        "PASS  exact ID/hash parity across stores"
        if sync_ok
        else "FAIL  release mismatch detected"
    )
except Exception as exc:
    results["8_release_sync"] = f"ERROR {exc}"

# ── Check 9: clinical sign-off & chunk pinning gate ────────────────────────
try:
    if not POLICY_ARTIFACT_PATH.exists():
        results["9_clinical_signoff"] = f"FAIL  policy file missing ({POLICY_ARTIFACT_PATH.name})"
    else:
        policy_rules = json.loads(POLICY_ARTIFACT_PATH.read_text(encoding="utf-8"))
        unsigned = [
            r.get("assertion_id")
            for r in policy_rules
            if r.get("reviewed_by") in (None, "", "pending_clinical_signoff", "TBD", "prototype")
            or not r.get("provenance", {}).get("chunk_id")
        ]
        if not policy_rules:
            results["9_clinical_signoff"] = "FAIL  no policy rules in artifact"
        elif unsigned or globals().get("PINNED_NSAID_CHUNK_ID") is None:
            results["9_clinical_signoff"] = (
                f"FAIL  missing clinical sign-off or chunk pin ({len(unsigned)} unsigned rule(s))"
            )
        else:
            results["9_clinical_signoff"] = (
                f"PASS  all {len(policy_rules)} policy rule(s) signed off and pinned"
            )
except Exception as exc:
    results["9_clinical_signoff"] = f"ERROR {exc}"

# ── Print summary ──────────────────────────────────────────────────────────
print("\n" + "=" * 76)
print("VERIFICATION SUMMARY")
print("=" * 76)
for key, val in results.items():
    if val.startswith("PASS"):
        icon = "\u2713"
    elif val.startswith("SKIP"):
        icon = "~"
    else:
        icon = "\u2717"
    print(f"  {icon}  [{key}]  {val}")
print("=" * 76)